# 🏛️ Google Cloud Enterprise Metadata Assessment & AI Inventory
## Framework Corporativo de Auditoria de Dados, Grafos, IA e Governança

> **🛡️ CONFORMIDADE ESTRITA COM A LGPD (ZERO DATA / METADATA-ONLY)**:  
> Este notebook foi desenvolvido para rodar com segurança no **Colab do BigQuery / Colab Enterprise**.  
> Ele **NÃO** lê dados transacionais, registros ou linhas de tabelas de clientes.  
> Todas as coletas são limitadas a **metadados arquiteturais**, DDLs de esquemas, dicionários de campos, definições de grafos (GQL), modelos de IA (Vertex/BQML), personas de agentes conversacionais, scans do Knowledge Catalog (Dataplex) e topologias de pipelines de carga.

---
### 🎯 Escopo das 8 Camadas Auditadas:
1. **BigQuery Core**: Datasets, tabelas, views, materialized views, esquemas e descrições de negócio.
2. **Property Graphs**: Grafos de propriedades (`INFORMATION_SCHEMA.PROPERTY_GRAPHS`) com nós, arestas e chaves.
3. **BigQuery ML**: Modelos de Machine Learning criados no BigQuery (`INFORMATION_SCHEMA.MODELS`).
4. **Vertex AI**: Model Registry (modelos, containers, versões), Endpoints, Datasets de IA e Pipelines de MLOps.
5. **Cloud Composer / Apache Airflow**: Ambientes gerenciados, versões (Composer 2/3, Airflow 2), workloads e pacotes PyPI.
6. **BigQuery Data Agents**: Agentes conversacionais da API `geminidataanalytics`, prompts executivos e *verified queries*.
7. **Knowledge Catalog & Dataplex**: Scans de perfil de dados (`DATA_PROFILE`), documentação e governança.
8. **Linhagem, Pipelines & Bancos**: Data Lineage API, Dataflow, Datastream, Dataproc, Cloud SQL e Spanner DDL.

### 📦 1. Dependências do Ambiente
No BigQuery Studio Colab / Colab Enterprise, as principais bibliotecas já vêm pré-instaladas. Execute a célula abaixo para garantir as dependências visuais e de manipulação.

In [ ]:
# @title Instalação rápida de dependências adicionais (se necessário)
!pip install -q requests pandas matplotlib seaborn tabulate

import os
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown, HTML

# Estilo visual elegante para gráficos corporativos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print('✅ Ambiente pronto!')

### ⚙️ 2. Parâmetros de Execução: Escopo da Organização & Destino no GCS
Configure abaixo o escopo da avaliação. Você pode avaliar **todos os projetos da Organização GCP automaticamente** (sem precisar rodar o notebook um por um), avaliar apenas o projeto atual ou fornecer uma lista específica de projetos.

In [ ]:
# @title ⚙️ Formulário de Configuração Multi-Projeto & Organização (Colab Form)
# @markdown **1. Modo de Escopo da Avaliação:**
# @markdown - `ORGANIZATION`: Avalia todos os projetos ativos da Organização GCP automaticamente (Recomendado)
# @markdown - `CURRENT_PROJECT`: Avalia apenas o projeto host onde este notebook está rodando
# @markdown - `CUSTOM_PROJECTS`: Avalia uma lista específica de projetos informada abaixo
ASSESSMENT_SCOPE = "ORGANIZATION"  # @param ["ORGANIZATION", "CURRENT_PROJECT", "CUSTOM_PROJECTS"]

# @markdown **2. ID da Organização GCP (opcional):**
# @markdown *Se deixar vazio no modo ORGANIZATION, busca automaticamente todos os projetos ativos acessíveis.*
ORGANIZATION_ID = ""  # @param {type:"string"}

# @markdown **3. Projetos específicos (usado quando o modo for CUSTOM_PROJECTS ou CURRENT_PROJECT):**
# @markdown *Exemplo: `proj-analytics, proj-data-lake, proj-crm` (deixe vazio para auto-descoberta)*
CUSTOM_PROJECTS_LIST = ""  # @param {type:"string"}

# @markdown **4. Destino no Google Cloud Storage (GCS) para salvar os resultados:**
# @markdown *Exemplo: `gs://meu-bucket-assessment/resultados_2026`*
GCS_OUTPUT_URI = "gs://seu-bucket-assessment/output"  # @param {type:"string"}

# @markdown **5. Filtro opcional de Datasets BigQuery:** (deixe vazio para avaliar TODOS os datasets)
DATASET_FILTER_RAW = ""  # @param {type:"string"}

# @markdown **6. Paralelismo Concorrente (Projetos em paralelo simultaneamente):**
# @markdown *Configurado por padrão para 4 workers simultâneos (mínimo de 4 em 4). Aumente para 6 ou 8 se desejar maior velocidade.*
MAX_WORKERS = 4  # @param {type:"integer"}

# Processamento dos parâmetros de entrada
PROJECT_IDS = [p.strip() for p in CUSTOM_PROJECTS_LIST.split(',') if p.strip()] or None
DATASET_FILTER = [d.strip() for d in DATASET_FILTER_RAW.split(',') if d.strip()] or None
OUTPUT_DIR = "output"

print(f"🏢 Escopo Selecionado: {ASSESSMENT_SCOPE}")
print(f"🏢 Organização: {ORGANIZATION_ID or 'Global / Auto-Detectada'}")
print(f"🎯 Projetos Informados: {PROJECT_IDS or 'Auto-Descoberta via Organização'}")
print(f"⚡ Paralelismo Concorrente: {MAX_WORKERS} workers simultâneos")
print(f"☁️ Destino GCS: {GCS_OUTPUT_URI}")
print(f"🎯 Datasets Filtrados: {DATASET_FILTER or 'TODOS os Datasets'}")

### 🛠️ 3. Engine Corporativa de Extração de Metadados (Self-Contained)
Esta célula contém a lógica modular completa de extração das 8 camadas, compatível com o Colab do BigQuery e protegida com os guardrails de LGPD.

In [ ]:
# @title Carregamento da Engine do Extrator e Business Assessment
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
===============================================================================
Módulo Corporativo: GCP Business Assessment Engine
===============================================================================
Analisa os metadados extraídos do ecossistema Google Cloud (BigQuery, Grafos,
Vertex AI, Composer, Data Agents, Dataplex, Linhagem e Bancos) para gerar:
  1. Diagnóstico e Mapeamento Semântico de Domínios de Negócio
  2. Formulação Estruturada de Casos de Negócio (Business Cases & Deliverables)
  3. Matriz de Priorização (Impacto no Negócio vs. Complexidade / Quick Wins)
  4. Avaliação de Prontidão de Dados para IA (Data Readiness & Grounding)
  5. Roadmap Estratégico por Ondas de Entrega (Ondas 1, 2 e 3)
  6. Artefatos Executivos em Markdown (.md) e JSON estruturado (.json)

CONFORMIDADE COM A LGPD:
  - 100% Baseado em Metadados Arquiteturais e Dicionários de Campos.
  - Zero Leitura de Dados de Linhas / Zero PII.
===============================================================================
"""

import os
import json
import re
from typing import Dict, List, Any, Optional, Tuple
from datetime import datetime, timezone

class GCPBusinessAssessmentEngine:
    """
    Motor de análise estratégica e geração de Casos de Negócio a partir de metadados GCP.
    """

    DOMAIN_RULES = {
        "Comercial & Vendas (GTM)": {
            "keywords": [
                "venda", "vendas", "sales", "pedido", "pedidos", "order", "orders",
                "faturamento", "receita", "revenue", "sell_in", "sell_out", "cliente",
                "customer", "crm", "loja", "store", "pdv", "pos", "canal", "channel",
                "representante", "rep", "meta", "comissao", "carteira", "checkout"
            ],
            "description": "Gestão de receita, conversão de vendas, sell-in/sell-out e produtividade comercial.",
            "default_benchmark": "Mercadolibre (Ad Manager Kairós) / Falabella: Text-to-SQL e copiloto comercial com BigQuery + Gemini.",
            "icon": "📈"
        },
        "Financeiro, RGM & Pricing": {
            "keywords": [
                "preco", "price", "pricing", "margem", "margin", "desconto", "discount",
                "custo", "cost", "lucro", "profit", "contabil", "fiscal", "tributo",
                "tax", "dre", "balanco", "pagamento", "payment", "cobranca", "inadimplencia",
                "aging", "recebiveis", "rgm", "tarifa"
            ],
            "description": "Rentabilidade, Revenue Growth Management, auditoria de descontos e fluxo de caixa.",
            "default_benchmark": "Mercadolibre: Auditoria retrospectiva de promoções e diagnóstico de ROI com BigQuery + Looker.",
            "icon": "💰"
        },
        "Supply Chain & Logística": {
            "keywords": [
                "estoque", "stock", "inventory", "armazem", "warehouse", "cd",
                "centro_distribuicao", "frete", "freight", "fob", "cif", "ruptura",
                "out_of_stock", "logistica", "shipping", "supply", "fornecedor",
                "supplier", "compra", "purchasing", "sku", "lead_time", "rastreamento"
            ],
            "description": "Visibilidade de inventário ponta a ponta, prevenção de rupturas e calibração logística.",
            "default_benchmark": "Weg Equipamentos (Projeto WENDi): Ingestão de eventos e resposta automatizada com Gemini + BigQuery.",
            "icon": "📦"
        },
        "Governança, Compliance & LGPD": {
            "keywords": [
                "cpf", "cnpj", "kyc", "consentimento", "consent", "bloqueio",
                "impedimento", "fraude", "fraud", "risco", "risk", "auditoria",
                "audit", "lgpd", "gdpr", "politica", "termo", "privacidade",
                "bolsa_familia", "beneficiario", "pep", "aml", "compliance", "seguranca"
            ],
            "description": "Conformidade regulatória, triagem de risco de clientes, privacidade de dados e mitigação legal.",
            "default_benchmark": "IQ Outsourcing: Validação automatizada de regras regulatórias e triagem médico-cadastral (85% automação).",
            "icon": "🛡️"
        },
        "Marketing & Retenção de Clientes": {
            "keywords": [
                "campanha", "campaign", "lead", "leads", "conversao", "conversion",
                "churn", "retencao", "retention", "nps", "engajamento", "click",
                "cliques", "ad", "marketing", "trafego", "audience", "audiencia",
                "segmentacao", "cluster", "promocao"
            ],
            "description": "Aquisição de clientes, redução de churn, personalização preditiva e otimização de campanhas.",
            "default_benchmark": "Magazine Luiza: Deduplicação e recomendação com Vertex AI Vector Search e BQML.",
            "icon": "🎯"
        },
        "Operações & Manufatura": {
            "keywords": [
                "producao", "production", "linha", "fabrica", "factory", "planta",
                "plant", "oee", "setup", "maquina", "machine", "sensor", "iot",
                "equipamento", "maintenance", "manutencao", "lote", "batch", "operacao"
            ],
            "description": "Eficiência fabril, sequenciamento de linha, redução de tempos de parada e manutenção preditiva.",
            "default_benchmark": "Weg Equipamentos: Otimização de linha fabril com OR-Tools + BigQuery telemetry.",
            "icon": "⚙️"
        },
        "IA Analítica & Decisão Conversacional": {
            "keywords": [
                "agent", "gemini", "ia", "ai", "ml", "modelo", "model", "prompt",
                "embedding", "vector", "conversational", "rag", "analytics"
            ],
            "description": "Cockpits executivos inteligentes, copilotos corporativos em linguagem natural e automação cognitiva.",
            "default_benchmark": "Falabella Mallplaza: Agente conversacional Text-to-SQL reduzindo tempo de resposta de 3 dias para minutos.",
            "icon": "🤖"
        }
    }

    def __init__(self, manifest: Dict[str, Any]):
        self.manifest = manifest
        self.header = manifest.get("metadata_header", {})
        self.bq_data = manifest.get("bigquery", {})
        self.vertex_data = manifest.get("vertex_ai", {})
        self.data_agents = manifest.get("data_agents", {}).get("data_agents", [])
        self.composer_envs = manifest.get("composer_airflow", {}).get("environments", [])
        self.lineage_data = manifest.get("lineage_and_pipelines", {})
        self.db_data = manifest.get("operational_databases", {})
        self.tables = self.bq_data.get("tables_and_views", [])
        self.property_graphs = self.bq_data.get("property_graphs", [])
        self.ml_models = self.bq_data.get("ml_models", [])

    def classify_table_domain(self, table: Dict[str, Any]) -> str:
        """Classifica uma tabela em um domínio de negócio com base no nome e descrições."""
        name_text = f"{table.get('dataset_id', '')} {table.get('table_name', '')} {table.get('description', '')}".lower()
        col_names = " ".join([c.get("column_name", "").lower() for c in table.get("columns", [])])
        full_text = f"{name_text} {col_names}"

        scores = {}
        for domain, info in self.DOMAIN_RULES.items():
            score = 0
            for kw in info["keywords"]:
                if re.search(r'\b' + re.escape(kw) + r'\b', full_text):
                    score += 2
                elif kw in full_text:
                    score += 1
            scores[domain] = score

        best_domain = max(scores, key=scores.get)
        if scores[best_domain] > 0:
            return best_domain
        return "Comercial & Vendas (GTM)"

    def evaluate_table_readiness(self, table: Dict[str, Any]) -> Tuple[float, str]:
        """Calcula o score de prontidão para IA e analítica da tabela."""
        cols = table.get("columns_count", 0)
        doc_cols = table.get("documented_columns_count", 0)
        profile_active = table.get("dataplex_profile_scan_active", False)

        doc_pct = (doc_cols / cols * 100) if cols > 0 else 0.0
        score = (doc_pct * 0.7) + (30.0 if profile_active else 0.0)

        if score >= 75:
            status = "🟢 Pronto para IA (Grounding Aprovado)"
        elif score >= 40:
            status = "🟡 Parcial (Requer Documentação Adicional)"
        else:
            status = "🔴 Gaps Críticos de Metadados"
        return round(score, 1), status

    def generate_assessment(self) -> Dict[str, Any]:
        """Executa a formulação dos Casos de Negócio corporativos."""
        # 1. Agrupar tabelas por domínio
        tables_by_domain: Dict[str, List[Dict[str, Any]]] = {}
        for t in self.tables:
            dom = self.classify_table_domain(t)
            if dom not in tables_by_domain:
                tables_by_domain[dom] = []
            tables_by_domain[dom].append(t)

        if not tables_by_domain:
            tables_by_domain["Comercial & Vendas (GTM)"] = []

        business_cases = []
        case_idx = 1

        for domain, domain_tables in tables_by_domain.items():
            if not domain_tables:
                continue

            dom_info = self.DOMAIN_RULES.get(domain, self.DOMAIN_RULES["Comercial & Vendas (GTM)"])
            table_names = [f"{t.get('project_id', '')}.{t.get('dataset_id', '')}.{t.get('table_name', '')}" for t in domain_tables]
            total_cols = sum(t.get("columns_count", 0) for t in domain_tables)
            total_doc = sum(t.get("documented_columns_count", 0) for t in domain_tables)
            doc_pct = round((total_doc / total_cols * 100), 1) if total_cols > 0 else 0.0
            profiled_count = sum(1 for t in domain_tables if t.get("dataplex_profile_scan_active"))

            readiness_score = round((doc_pct * 0.7) + ((profiled_count / len(domain_tables)) * 30.0), 1)
            readiness_status = "🟢 Pronto para IA" if readiness_score >= 75 else ("🟡 Parcial" if readiness_score >= 40 else "🔴 Gaps Críticos")

            if "Governança" in domain or "Compliance" in domain or any("bloqueio" in t.lower() or "cpf" in t.lower() for t in table_names):
                bc = {
                    "case_id": f"BC-{case_idx:02d}",
                    "domain": domain,
                    "title": "Módulo de Triagem Regulatória, KYC e Bloqueio Preventivo (LGPD)",
                    "problem_statement": "Risco de exposição a multas regulatórias, fraudes e concessão de serviços a cadastros impedidos ou sem conformidade de consentimento.",
                    "deliverable": "Módulo de Conformidade e Proteção de Dados com reconciliação automatizada via BigQuery e Dataplex.",
                    "business_impact": "Mitigação total de riscos legais e corte imediato de custos operacionais com processos irregulares.",
                    "impact_level": "ALTO",
                    "complexity_level": "BAIXA",
                    "timeframe_weeks": "4 semanas",
                    "quadrant": "Quick Win",
                    "wave": "Onda 1",
                    "supporting_tables": table_names[:6],
                    "readiness_score": readiness_score,
                    "readiness_status": readiness_status,
                    "benchmark": "IQ Outsourcing: 85% de automação de conformidade e triagem médico-cadastral com Gemini + BigQuery.",
                    "target_gcp_stack": "BigQuery (SQL Joins) + Dataplex Knowledge Catalog + Cloud Run"
                }
            elif "Supply" in domain or "Logística" in domain or any("estoque" in t.lower() for t in table_names):
                bc = {
                    "case_id": f"BC-{case_idx:02d}",
                    "domain": domain,
                    "title": "Torre de Controle Logística: Visibilidade de Rupturas CD/Loja e Calibração FOB",
                    "problem_statement": "Descompasso de inventário entre depósitos centrais e pontos de distribuição, gerando faturamento perdido e duplicação de pedidos no S&OP.",
                    "deliverable": "Módulo de Monitoramento de Rupturas e Reconciliação Física no BigQuery + Looker.",
                    "business_impact": "Redução de até 40% nas perdas por ruptura de estoque e eliminação de estoque fantasma.",
                    "impact_level": "ALTO",
                    "complexity_level": "BAIXA",
                    "timeframe_weeks": "4 semanas",
                    "quadrant": "Quick Win",
                    "wave": "Onda 1",
                    "supporting_tables": table_names[:6],
                    "readiness_score": readiness_score,
                    "readiness_status": readiness_status,
                    "benchmark": "Weg Equipamentos: Ingestão de eventos e respostas automatizadas com Gemini e BigQuery.",
                    "target_gcp_stack": "BigQuery + Dataform + Looker Studio"
                }
            elif "Financeiro" in domain or "Pricing" in domain:
                bc = {
                    "case_id": f"BC-{case_idx:02d}",
                    "domain": domain,
                    "title": "Revenue Growth Management: Diagnóstico de ROI Promocional e Copiloto de Margens",
                    "problem_statement": "Concessão agressiva de descontos e políticas de preço sem análise retrospectiva de elasticidade, erodindo as margens operacionais.",
                    "deliverable": "Módulo de Auditoria de Descontos e Simulador de Elasticidade de Preço com BigQuery ML.",
                    "business_impact": "Recuperação de 2% a 5% de margem líquida através da eliminação de descontos ineficientes.",
                    "impact_level": "ALTO",
                    "complexity_level": "MÉDIA",
                    "timeframe_weeks": "6 semanas",
                    "quadrant": "Aposta Estratégica",
                    "wave": "Onda 2",
                    "supporting_tables": table_names[:6],
                    "readiness_score": readiness_score,
                    "readiness_status": readiness_status,
                    "benchmark": "Mercadolibre: Diagnóstico de campanhas com análise de incrementabilidade e margem.",
                    "target_gcp_stack": "BigQuery ML + Looker + Vertex AI Model Registry"
                }
            elif "Marketing" in domain or "Retenção" in domain:
                bc = {
                    "case_id": f"BC-{case_idx:02d}",
                    "domain": domain,
                    "title": "Motor de Personalização & Prevenção Preditiva de Churn de Clientes",
                    "problem_statement": "Perda de clientes e ineficiência em disparos de campanhas genéricas sem segmentação comportamental precisa.",
                    "deliverable": "Pipeline de Clusterização de Perfis e Score de Propensão com Vertex AI e Vector Search.",
                    "business_impact": "Aumento de 15% na taxa de conversão e redução de 20% no churn de clientes prioritários.",
                    "impact_level": "MÉDIO",
                    "complexity_level": "MÉDIA",
                    "timeframe_weeks": "6 semanas",
                    "quadrant": "Melhoria Tática",
                    "wave": "Onda 2",
                    "supporting_tables": table_names[:6],
                    "readiness_score": readiness_score,
                    "readiness_status": readiness_status,
                    "benchmark": "Magazine Luiza: Deduplicação e depara de produtos com Embeddings no Vertex AI.",
                    "target_gcp_stack": "BigQuery + Vertex AI Vector Search + Cloud Run"
                }
            else:
                bc = {
                    "case_id": f"BC-{case_idx:02d}",
                    "domain": domain,
                    "title": "Módulo Comercial de Reposição Inteligente e Next Best Action no CRM",
                    "problem_statement": "Vendedores e representantes gastam horas em planilhas operacionais em vez de focar no fechamento de pedidos de alto valor.",
                    "deliverable": "Assistente de Recomendação de Vendas e Sugestão de Reposição por PDV conectado ao CRM.",
                    "business_impact": "Crescimento de 8% a 12% no sell-in recorrente e liberação de 30% do tempo dos gerentes comerciais.",
                    "impact_level": "ALTO",
                    "complexity_level": "BAIXA",
                    "timeframe_weeks": "4 semanas",
                    "quadrant": "Quick Win",
                    "wave": "Onda 1",
                    "supporting_tables": table_names[:6],
                    "readiness_score": readiness_score,
                    "readiness_status": readiness_status,
                    "benchmark": "Falabella Mallplaza: Text-to-SQL reduzindo o tempo de consulta de 3 dias para segundos.",
                    "target_gcp_stack": "BigQuery + Data Agents (Gemini Data Analytics) + Apigee"
                }

            business_cases.append(bc)
            case_idx += 1

        if self.data_agents or self.property_graphs:
            graph_names = [g.get("property_graph_name", "") for g in self.property_graphs]
            bc_ai = {
                "case_id": f"BC-{case_idx:02d}",
                "domain": "IA Analítica & Decisão Conversacional",
                "title": "Cockpit Executivo Conversacional: BigQuery Data Agents Grounded em Grafos (GQL)",
                "problem_statement": "Decisões estratégicas da diretoria dependem de filas de chamados para times de BI, demorando dias para respostas ad-hoc complexas.",
                "deliverable": "Rede de Data Agents do BigQuery conectados a Grafos Semânticos (GQL) para consultas corporativas instantâneas.",
                "business_impact": "Respostas executivas com 100% de acurácia em menos de 5 segundos, reduzindo a carga do time de dados em 65%.",
                "impact_level": "ALTO",
                "complexity_level": "MÉDIA",
                "timeframe_weeks": "6 semanas",
                "quadrant": "Aposta Estratégica",
                "wave": "Onda 3",
                "supporting_tables": [f"Grafos: {', '.join(graph_names[:3])}"] if graph_names else ["Tabelas Analíticas Silver/Gold"],
                "readiness_score": 85.0 if self.data_agents else 60.0,
                "readiness_status": "🟢 Pronto para IA (Agentes Mapeados)",
                "benchmark": "Google Cloud Data Agent Kit: Agentes C-Level com suporte a streaming, grounding semântico e raciocínio analítico.",
                "target_gcp_stack": "BigQuery Data Agents + Vertex AI + Cloud Run"
            }
            business_cases.append(bc_ai)
            case_idx += 1

        profile_pct = self.header.get("global_kpis", {}).get("profile_coverage_pct", 0)
        if profile_pct < 100:
            bc_gov = {
                "case_id": f"BC-{case_idx:02d}",
                "domain": "Governança, Compliance & LGPD",
                "title": "Módulo de Perfilamento Contínuo e Enriquecimento Semântico no Knowledge Catalog",
                "problem_statement": "Tabelas e views sem documentação ou perfilamento ativo criam risco de alucinação de IA e perda de linhagem de dados.",
                "deliverable": "Esteira automatizada de Data Profile Scans e geração de descrições de negócio no Dataplex Catalog.",
                "business_impact": "Garantia de 100% de conformidade com auditoria de dados e aceleração de novos casos de IA.",
                "impact_level": "MÉDIO",
                "complexity_level": "BAIXA",
                "timeframe_weeks": "4 semanas",
                "quadrant": "Melhoria Tática",
                "wave": "Onda 1",
                "supporting_tables": ["Tabelas Gold & Silver não perfiladas"],
                "readiness_score": float(profile_pct),
                "readiness_status": "🟡 Requer Scans de Perfilamento",
                "benchmark": "Knowledge Catalog Data Insights: scans automatizados sem impacto na performance de produção.",
                "target_gcp_stack": "Dataplex Data Profile API + Knowledge Catalog"
            }
            business_cases.append(bc_gov)

        quick_wins = [c for c in business_cases if c["quadrant"] == "Quick Win"]
        strategic_bets = [c for c in business_cases if c["quadrant"] == "Aposta Estratégica"]
        tactical_improvements = [c for c in business_cases if c["quadrant"] == "Melhoria Tática"]
        long_term = [c for c in business_cases if c["quadrant"] == "Longo Prazo"]

        wave_1 = [c for c in business_cases if c["wave"] == "Onda 1"]
        wave_2 = [c for c in business_cases if c["wave"] == "Onda 2"]
        wave_3 = [c for c in business_cases if c["wave"] == "Onda 3"]

        assessment_result = {
            "business_assessment_header": {
                "generated_at": datetime.now(timezone.utc).isoformat(),
                "total_business_cases": len(business_cases),
                "quick_wins_count": len(quick_wins),
                "strategic_bets_count": len(strategic_bets),
                "tactical_count": len(tactical_improvements),
                "domains_evaluated": list(tables_by_domain.keys()),
                "compliance_status": "LGPD Safe - 100% Metadata-Driven"
            },
            "business_cases": business_cases,
            "prioritization_matrix": {
                "quick_wins": quick_wins,
                "strategic_bets": strategic_bets,
                "tactical_improvements": tactical_improvements,
                "long_term_projects": long_term
            },
            "wave_roadmap": {
                "wave_1_quick_wins": {
                    "phase": "Onda 1: Quick Wins & Redução Imediata de Custo",
                    "timeframe": "Meses 1-2 (4 semanas por entregável)",
                    "cases": wave_1
                },
                "wave_2_strategic": {
                    "phase": "Onda 2: Eficiência Operacional & Integração Estruturante",
                    "timeframe": "Meses 3-4 (6 semanas por entregável)",
                    "cases": wave_2
                },
                "wave_3_ai_advanced": {
                    "phase": "Onda 3: IA Preditiva, Agentes Conversacionais & Vantagem Competitiva",
                    "timeframe": "Meses 5-6 (8 semanas por entregável)",
                    "cases": wave_3
                }
            }
        }
        return assessment_result

    def generate_markdown_report(self, assessment_data: Dict[str, Any]) -> str:
        """Gera o relatório executivo C-Level em Markdown estruturado."""
        cases = assessment_data.get("business_cases", [])
        matrix = assessment_data.get("prioritization_matrix", {})
        roadmap = assessment_data.get("wave_roadmap", {})
        header = assessment_data.get("business_assessment_header", {})

        lines = [
            "# 💼 Business Assessment & Casos de Negócio Estratégicos GCP",
            "",
            f"**Data da Avaliação:** `{header.get('generated_at')}`  ",
            f"**Total de Casos de Negócio Estruturados:** `{header.get('total_business_cases')}`  ",
            f"**Quick Wins Identificados (Alto Impacto / Baixa Complexidade):** `{header.get('quick_wins_count')}`  ",
            f"**Conformidade de Governança:** `{header.get('compliance_status')}`  ",
            "",
            "---",
            "",
            "## 🎯 1. Resumo Executivo das Oportunidades de Negócio",
            "",
            "A análise arquitetural e semântica dos metadados identificou oportunidades concretas para acelerar o retorno sobre investimento (ROI), alavancar inteligência artificial corporativa e modernizar os processos operacionais sobre a plataforma **Google Cloud**.",
            "",
            "| ID | Domínio de Negócio | Caso de Uso / Entregável | Impacto | Complexidade | Prazo | Quadrante | Prontidão IA |",
            "|---|---|---|---|---|---|---|---|"
        ]

        for c in cases:
            lines.append(
                f"| `{c['case_id']}` | **{c['domain']}** | {c['title']} | **{c['impact_level']}** | {c['complexity_level']} | {c['timeframe_weeks']} | **{c['quadrant']}** | {c['readiness_status']} |"
            )

        lines.extend([
            "",
            "---",
            "",
            "## 🏆 2. Matriz de Priorização de Casos de Negócio (Impacto vs. Complexidade)",
            "",
            "```",
            "   ALTO |-----------------------------------------------------------------------|",
            "        | 🟢 QUADRANTE 1: QUICK WINS           | 🔵 QUADRANTE 2: APOSTAS ESTRATÉGICAS   |",
            f"        | - {len(matrix.get('quick_wins', []))} Casos (Prazo: 4 semanas)       | - {len(matrix.get('strategic_bets', []))} Casos (Prazo: 6-8 semanas)    |",
            "        | Prioridade de Execução Imediata       | Alto Retorno / Integração Estruturante|",
            " I      |---------------------------------------|---------------------------------------|",
            " M      | 🟡 QUADRANTE 3: MELHORIAS TÁTICAS     | ⚪ QUADRANTE 4: LONGO PRAZO            |",
            f" P      | - {len(matrix.get('tactical_improvements', []))} Casos (Automação Contínua)   | - {len(matrix.get('long_term_projects', []))} Casos (Despriorizados)           |",
            " A      | Eficiência e Higienização de Dados    | Alto Esforço / Retorno Incremental    |",
            " C      |-----------------------------------------------------------------------|",
            " T      BAIXA <----------------------- COMPLEXIDADE ----------------------> ALTA",
            " O",
            "```",
            "",
            "---",
            "",
            "## 🌊 3. Roadmap Estratégico de Implementação por Ondas",
            ""
        ])

        for wave_key, wave_info in roadmap.items():
            lines.extend([
                f"### 📍 {wave_info['phase']} ({wave_info['timeframe']})",
                ""
            ])
            for c in wave_info.get("cases", []):
                lines.extend([
                    f"#### 🎯 [{c['case_id']}] {c['title']}",
                    f"- **Domínio:** {c['domain']}",
                    f"- **Diagnóstico / Dor de Negócio:** {c['problem_statement']}",
                    f"- **Entregável Específico:** {c['deliverable']}",
                    f"- **Ganho de Negócio / Valor:** {c['business_impact']}",
                    f"- **Prontidão dos Dados:** {c['readiness_status']} (Score: {c['readiness_score']}%)",
                    f"- **Ativos de Dados Vinculados:** `{', '.join(c['supporting_tables']) if c['supporting_tables'] else 'Tabelas do Domínio'}`",
                    f"- **Benchmark de Mercado:** {c['benchmark']}",
                    f"- **Tecnologia GCP Mínima:** `{c['target_gcp_stack']}`",
                    ""
                ])

        lines.extend([
            "---",
            "",
            "## 🛡️ 4. Diretrizes de Governança & Conformidade LGPD",
            "",
            "1. **Zero Exposição de Linhas**: Este assessment foi gerado exclusivamente a partir de DDLs, schemas, grafos, modelos e metadados de catálogo, garantindo zero violação de privacidade de clientes.",
            "2. **Grounding Seguro para IA**: Para habilitar agentes conversacionais (BigQuery Data Agents) e modelos preditivos, todas as tabelas devem possuir descrições semânticas detalhadas em nível de coluna e scans de perfil de dados no Knowledge Catalog.",
            ""
        ])

        return "\n".join(lines)

    def export_artifacts(self, output_dir: str) -> Tuple[str, str, Dict[str, Any]]:
        """Gera e salva os arquivos do Business Assessment na pasta de saída."""
        os.makedirs(output_dir, exist_ok=True)
        assessment_data = self.generate_assessment()
        markdown_content = self.generate_markdown_report(assessment_data)

        md_path = os.path.join(output_dir, "business_assessment_cases.md")
        json_path = os.path.join(output_dir, "business_assessment_cases.json")

        with open(md_path, "w", encoding="utf-8") as f:
            f.write(markdown_content)

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(assessment_data, f, indent=2, ensure_ascii=False)

        print(f"💼 Business Assessment Markdown salvo em: {md_path}", flush=True)
        print(f"📊 Business Assessment JSON salvo em: {json_path}", flush=True)

        return md_path, json_path, assessment_data


#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
===============================================================================
Google Cloud Enterprise Metadata Assessment Extractor
===============================================================================
Módulo corporativo para avaliação (assessment) e extração padronizada de
metadados de ambientes Google Cloud, com conformidade estrita com a LGPD.

Camadas Inspecionadas:
1. BigQuery Core (Datasets, Tabelas, Views, Colunas, Tipos, Descrições de Negócio)
2. BigQuery Property Graphs (Nós, Arestas, Chaves, Propriedades, DDLs GQL)
3. BigQuery ML (Modelos de Machine Learning no BQ, Algoritmos, Features, Labels)
4. Vertex AI (Model Registry, Endpoints, Datasets de IA, Pipelines de MLOps)
5. Cloud Composer / Apache Airflow (Ambientes, Versões, Workloads, DAGs, Sizing)
6. BigQuery Data Agents (Agentes Conversacionais, System Instructions, Golden Queries)
7. Knowledge Catalog / Dataplex (Data Profile Scans, Documentation Scans, Governança)
8. Linhagem & Pipelines (Data Lineage API, Dataflow, Datastream, Dataproc)
9. Bancos Operacionais (Cloud SQL instâncias/bancos, Spanner DDLs, Vetores e Grafos)

SEGURANÇA & LGPD:
- ZERO DATA EXTRACTION: Nenhuma query 'SELECT *' em dados de clientes.
- Apenas esquemas, DDLs, configurações, estatísticas de storage e metadados de governança.
===============================================================================
"""

import os
import re
import sys
import json
import time
import subprocess
import datetime
import threading
import concurrent.futures
from typing import Dict, List, Any, Optional, Tuple

import requests
import pandas as pd
import google.auth
import google.auth.transport.requests
from google.oauth2 import credentials as oauth2_creds
from google.cloud import bigquery
from google.cloud.exceptions import NotFound, Forbidden


class GCPAuthManager:
    """
    Gerenciador inteligente de autenticação GCP.
    Suporta Application Default Credentials (ADC) em Colab/Vertex e fallback
    automático para gcloud CLI token em execução local.
    """
    def __init__(self, project_id: Optional[str] = None):
        self.project_id = project_id
        self._credentials = None
        self._token = None
        self._init_auth()

    def _init_auth(self):
        try:
            creds, default_project = google.auth.default(
                scopes=["https://www.googleapis.com/auth/cloud-platform"]
            )
            auth_req = google.auth.transport.requests.Request()
            creds.refresh(auth_req)
            self._credentials = creds
            self._token = creds.token
            if not self.project_id:
                self.project_id = default_project
        except Exception as e:
            # Fallback para gcloud CLI token
            self._token = self._get_token_from_gcloud()
            if self._token:
                self._credentials = oauth2_creds.Credentials(token=self._token)
            else:
                raise RuntimeError(
                    f"Falha crítica de autenticação GCP. Execute 'gcloud auth login' e "
                    f"'gcloud auth application-default login'. Erro original: {e}"
                )

    def _get_token_from_gcloud(self) -> Optional[str]:
        try:
            out = subprocess.check_output(
                ["gcloud", "auth", "print-access-token"],
                stderr=subprocess.DEVNULL,
                text=True
            ).strip()
            return out if out else None
        except Exception:
            return None

    def _get_project_from_gcloud(self) -> Optional[str]:
        try:
            out = subprocess.check_output(
                ["gcloud", "config", "get-value", "project"],
                stderr=subprocess.DEVNULL,
                text=True
            ).strip()
            return out if out and out != "(unset)" else None
        except Exception:
            return None

    def get_token(self) -> str:
        if not self._token:
            self._init_auth()
        return self._token

    def get_headers(self) -> Dict[str, str]:
        return {
            "Authorization": f"Bearer {self.get_token()}",
            "Content-Type": "application/json"
        }

    def get_bigquery_client(self, project: Optional[str] = None) -> bigquery.Client:
        return bigquery.Client(
            project=project or self.project_id,
            credentials=self._credentials
        )


class LGPDGuardrail:
    """
    Guardião de Segurança e Conformidade com a LGPD.
    Garante que nenhum dado transacional ou segredo seja capturado ou vazado.
    """
    SECRET_PATTERNS = [
        re.compile(r"password", re.I),
        re.compile(r"secret", re.I),
        re.compile(r"token", re.I),
        re.compile(r"api[_-]?key", re.I),
        re.compile(r"private[_-]?key", re.I),
        re.compile(r"bearer", re.I),
        re.compile(r"credential", re.I),
    ]

    @classmethod
    def sanitize_env_vars(cls, env_dict: Dict[str, Any]) -> Dict[str, str]:
        """Anonimiza e mascara valores de variáveis de ambiente potencialmente sensíveis."""
        sanitized = {}
        for k, v in env_dict.items():
            is_sensitive = any(p.search(k) for p in cls.SECRET_PATTERNS)
            if is_sensitive:
                sanitized[k] = "[REDACTED_BY_LGPD_GUARDRAIL]"
            else:
                # Trunca se valor for excessivamente longo
                str_val = str(v)
                sanitized[k] = str_val if len(str_val) <= 120 else str_val[:117] + "..."
        return sanitized


class BigQueryMetadataExtractor:
    """
    Extrator de metadados do BigQuery:
    - Datasets, Schemata, Localizações, Labels
    - Tabelas, Views, Materialized Views, Schemas completos
    - Dicionário de Colunas e Descrições de Negócio
    - Property Graphs (GQL)
    - Modelos de Machine Learning (BQML)
    """
    def __init__(self, auth: GCPAuthManager, project_id: str):
        self.auth = auth
        self.project_id = project_id
        self.client = auth.get_bigquery_client(project_id)

    def extract_all(self, dataset_filter: Optional[List[str]] = None) -> Dict[str, Any]:
        print(f"[{self.project_id}] 🔍 [1/8 BigQuery] Iniciando varredura...", flush=True)
        datasets_meta = []
        tables_meta = []
        property_graphs_meta = []
        ml_models_meta = []

        try:
            all_datasets = list(self.client.list_datasets(project=self.project_id))
        except Exception as e:
            print(f"[{self.project_id}] ⚠️ [BigQuery] Falha ao listar datasets: {e}", flush=True)
            return {
                "datasets": [],
                "tables_and_views": [],
                "property_graphs": [],
                "ml_models": []
            }

        filtered_datasets = [
            d for d in all_datasets 
            if not dataset_filter or d.dataset_id in dataset_filter
        ]
        total_ds = len(filtered_datasets)
        print(f"[{self.project_id}]    📊 {total_ds} datasets identificados para auditoria.", flush=True)

        def _process_single_dataset(item: Tuple[int, Any]):
            idx, ds_ref = item
            ds_id = ds_ref.dataset_id
            try:
                ds = self.client.get_dataset(f"{self.project_id}.{ds_id}")
                ds_info = {
                    "project_id": self.project_id,
                    "dataset_id": ds_id,
                    "location": ds.location,
                    "description": ds.description or "",
                    "created_at": ds.created.isoformat() if ds.created else None,
                    "modified_at": ds.modified.isoformat() if ds.modified else None,
                    "labels": ds.labels or {}
                }
                tbls, graphs, models = self._extract_dataset_contents(ds_id)
                if tbls or graphs or models or total_ds <= 10:
                    print(f"[{self.project_id}]    ⏳ [{idx}/{total_ds}] Dataset '{ds_id}' -> {len(tbls)} tabelas/views, {len(graphs)} grafos", flush=True)
                return idx, ds_info, tbls, graphs, models
            except Exception as e:
                print(f"[{self.project_id}]    ⚠️ [{idx}/{total_ds}] Dataset '{ds_id}' -> Erro: {e}", flush=True)
                return idx, None, [], [], []

        ds_workers = min(4, total_ds) if total_ds > 1 else 1
        indexed_items = list(enumerate(filtered_datasets, 1))

        if ds_workers > 1:
            with concurrent.futures.ThreadPoolExecutor(max_workers=ds_workers) as ds_exec:
                results = list(ds_exec.map(_process_single_dataset, indexed_items))
        else:
            results = [_process_single_dataset(item) for item in indexed_items]

        # Ordena pelo índice original para consistência
        results.sort(key=lambda r: r[0])
        for _, ds_info, tbls, graphs, models in results:
            if ds_info:
                datasets_meta.append(ds_info)
            tables_meta.extend(tbls)
            property_graphs_meta.extend(graphs)
            ml_models_meta.extend(models)

        print(f"[{self.project_id}] ✅ [1/8 BigQuery] Concluído: {len(datasets_meta)} datasets, {len(tables_meta)} tabelas/views, "
              f"{len(property_graphs_meta)} grafos e {len(ml_models_meta)} modelos BQML.", flush=True)
        return {
            "datasets": datasets_meta,
            "tables_and_views": tables_meta,
            "property_graphs": property_graphs_meta,
            "ml_models": ml_models_meta
        }

    def _extract_dataset_contents(self, dataset_id: str) -> Tuple[List[Dict], List[Dict], List[Dict]]:
        tables_list = []
        graphs_list = []
        models_list = []

        # 1. Storage & contagem de linhas em lote via __TABLES__
        storage_info = {}
        try:
            sql_storage = f"SELECT table_id, row_count, size_bytes, type FROM `{self.project_id}.{dataset_id}.__TABLES__`"
            for r in self.client.query(sql_storage).result():
                storage_info[r["table_id"]] = {
                    "num_rows": r["row_count"],
                    "size_bytes": r["size_bytes"],
                    "raw_type": r["type"]
                }
        except Exception:
            pass

        # 2. Tabelas e Views via INFORMATION_SCHEMA.TABLES
        sql_tables = f"""
        SELECT 
            table_name, 
            table_type, 
            creation_time, 
            ddl
        FROM `{self.project_id}.{dataset_id}.INFORMATION_SCHEMA.TABLES`
        """
        table_dict = {}
        try:
            for r in self.client.query(sql_tables).result():
                table_dict[r["table_name"]] = {
                    "table_name": r["table_name"],
                    "table_type": r["table_type"],
                    "creation_time": r["creation_time"].isoformat() if r["creation_time"] else None,
                    "ddl": r.get("ddl", "")
                }
        except Exception:
            pass

        # Otimização: se o dataset estiver completamente vazio, pula queries de colunas e opções
        if not storage_info and not table_dict:
            return [], [], []

        # 3. Descrições de tabelas via TABLE_OPTIONS
        sql_tbl_options = f"""
        SELECT 
            table_name, 
            option_name, 
            option_value
        FROM `{self.project_id}.{dataset_id}.INFORMATION_SCHEMA.TABLE_OPTIONS`
        WHERE option_name IN ('description', 'friendly_name')
        """
        table_descriptions = {}
        try:
            for opt in self.client.query(sql_tbl_options).result():
                if opt["option_name"] == "description":
                    desc = opt["option_value"]
                    if desc.startswith('"') and desc.endswith('"'):
                        desc = desc[1:-1]
                    table_descriptions[opt["table_name"]] = desc
        except Exception:
            pass

        # 4. Colunas completas, tipos e partições via INFORMATION_SCHEMA.COLUMNS
        sql_cols = f"""
        SELECT 
            table_name, 
            column_name, 
            data_type, 
            is_nullable, 
            is_partitioning_column, 
            clustering_ordinal_position
        FROM `{self.project_id}.{dataset_id}.INFORMATION_SCHEMA.COLUMNS`
        ORDER BY table_name, ordinal_position
        """
        columns_by_table: Dict[str, List[Dict]] = {}
        partition_by_table: Dict[str, str] = {}
        cluster_by_table: Dict[str, List[str]] = {}
        try:
            for col in self.client.query(sql_cols).result():
                t_name = col["table_name"]
                c_name = col["column_name"]
                if t_name not in columns_by_table:
                    columns_by_table[t_name] = []
                
                if col.get("is_partitioning_column") == "YES":
                    partition_by_table[t_name] = c_name
                if col.get("clustering_ordinal_position") is not None:
                    if t_name not in cluster_by_table:
                        cluster_by_table[t_name] = []
                    cluster_by_table[t_name].append(c_name)

                columns_by_table[t_name].append({
                    "column_name": c_name,
                    "data_type": col["data_type"],
                    "is_nullable": col["is_nullable"],
                    "is_partitioning": col.get("is_partitioning_column") == "YES",
                    "description": ""  # preenchido no passo 5
                })
        except Exception:
            pass

        # 5. Descrições de colunas via COLUMN_FIELD_PATHS
        sql_col_desc = f"""
        SELECT 
            table_name, 
            field_path, 
            description
        FROM `{self.project_id}.{dataset_id}.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS`
        WHERE description IS NOT NULL AND description != ''
        """
        try:
            for c in self.client.query(sql_col_desc).result():
                t_name = c["table_name"]
                f_path = c["field_path"]
                desc_val = c["description"]
                if t_name in columns_by_table:
                    for col_item in columns_by_table[t_name]:
                        if col_item["column_name"] == f_path:
                            col_item["description"] = desc_val
        except Exception:
            pass

        # Consolidar todas as tabelas encontradas no dataset
        all_table_names = set(table_dict.keys()) | set(storage_info.keys())
        for t_name in sorted(all_table_names):
            t_info = table_dict.get(t_name, {})
            s_info = storage_info.get(t_name, {})
            
            t_type = t_info.get("table_type")
            if not t_type:
                raw_type = s_info.get("raw_type")
                t_type = "VIEW" if raw_type == 2 else ("BASE TABLE" if raw_type == 1 else "TABLE")

            columns = columns_by_table.get(t_name, [])
            total_cols = len(columns)
            documented_cols = sum(1 for c in columns if c.get("description", "").strip())
            doc_pct = round((documented_cols / total_cols * 100), 2) if total_cols > 0 else 0.0

            tbl_meta = {
                "project_id": self.project_id,
                "dataset_id": dataset_id,
                "table_name": t_name,
                "table_type": t_type,
                "description": table_descriptions.get(t_name, ""),
                "num_rows_estimated": s_info.get("num_rows", 0),
                "total_bytes_estimated": s_info.get("size_bytes", 0),
                "partition_column": partition_by_table.get(t_name),
                "clustering_fields": cluster_by_table.get(t_name),
                "columns_count": total_cols,
                "documented_columns_count": documented_cols,
                "documentation_coverage_pct": doc_pct,
                "columns": columns,
                "ddl": t_info.get("ddl", "")
            }
            tables_list.append(tbl_meta)

        # 6. Property Graphs via INFORMATION_SCHEMA.PROPERTY_GRAPHS
        sql_graphs = f"""
        SELECT 
            property_graph_name, 
            property_graph_schema, 
            ddl
        FROM `{self.project_id}.{dataset_id}.INFORMATION_SCHEMA.PROPERTY_GRAPHS`
        """
        try:
            for g in self.client.query(sql_graphs).result():
                graphs_list.append({
                    "project_id": self.project_id,
                    "dataset_id": dataset_id,
                    "property_graph_name": g["property_graph_name"],
                    "ddl": g["ddl"]
                })
        except Exception:
            pass

        # 7. BigQuery ML Models via INFORMATION_SCHEMA.MODELS
        sql_models = f"""
        SELECT 
            model_name, 
            model_type, 
            creation_time, 
            ddl
        FROM `{self.project_id}.{dataset_id}.INFORMATION_SCHEMA.MODELS`
        """
        try:
            for m in self.client.query(sql_models).result():
                models_list.append({
                    "project_id": self.project_id,
                    "dataset_id": dataset_id,
                    "model_name": m["model_name"],
                    "model_type": m["model_type"],
                    "creation_time": m["creation_time"].isoformat() if m["creation_time"] else None,
                    "ddl": m.get("ddl", "")
                })
        except Exception:
            pass

        return tables_list, graphs_list, models_list


class VertexAIExtractor:
    """
    Extrator de metadados do Vertex AI:
    - Model Registry (Modelos, Versões, Specs de Container, Métricas)
    - Endpoints (Modelos implantados, Split de tráfego, Recursos de máquina)
    - Datasets de IA (Tabular, Séries Temporais, Text Prompts, Links para BigQuery)
    - Pipelines de MLOps (PipelineJobs, Histórico de Execuções)
    """
    def __init__(self, auth: GCPAuthManager, project_id: str, locations: Optional[List[str]] = None):
        self.auth = auth
        self.project_id = project_id
        self.locations = locations or ["us-central1", "global"]

    def extract_all(self) -> Dict[str, Any]:
        print(f"[{self.project_id}] 🤖 [2/8 Vertex AI] Iniciando varredura...", flush=True)
        models_meta = []
        endpoints_meta = []
        datasets_meta = []
        pipelines_meta = []

        headers = self.auth.get_headers()

        for loc in self.locations:
            base_url = f"https://{loc}-aiplatform.googleapis.com/v1/projects/{self.project_id}/locations/{loc}"
            if loc == "global":
                base_url = f"https://aiplatform.googleapis.com/v1/projects/{self.project_id}/locations/global"

            loc_models_count = 0
            loc_datasets_count = 0

            # 1. Model Registry
            try:
                resp = requests.get(f"{base_url}/models", headers=headers, timeout=15)
                if resp.status_code == 200:
                    models = resp.json().get("models", [])
                    loc_models_count = len(models)
                    for m in models:
                        models_meta.append({
                            "project_id": self.project_id,
                            "name": m.get("name"),
                            "display_name": m.get("displayName"),
                            "version_id": m.get("versionId"),
                            "description": m.get("description", ""),
                            "create_time": m.get("createTime"),
                            "update_time": m.get("updateTime"),
                            "container_image": m.get("containerSpec", {}).get("imageUri"),
                            "artifact_uri": m.get("artifactUri"),
                            "location": loc,
                            "labels": m.get("labels", {})
                        })
            except Exception as e:
                pass

            # 2. Endpoints
            try:
                resp = requests.get(f"{base_url}/endpoints", headers=headers, timeout=15)
                if resp.status_code == 200:
                    endpoints = resp.json().get("endpoints", [])
                    for ep in endpoints:
                        deployed = ep.get("deployedModels", [])
                        endpoints_meta.append({
                            "project_id": self.project_id,
                            "name": ep.get("name"),
                            "display_name": ep.get("displayName"),
                            "description": ep.get("description", ""),
                            "create_time": ep.get("createTime"),
                            "update_time": ep.get("updateTime"),
                            "location": loc,
                            "traffic_split": ep.get("trafficSplit", {}),
                            "deployed_models_count": len(deployed),
                            "deployed_models": [
                                {
                                    "id": d.get("id"),
                                    "model": d.get("model"),
                                    "display_name": d.get("displayName"),
                                    "machine_type": d.get("dedicatedResources", {}).get("machineSpec", {}).get("machineType")
                                }
                                for d in deployed
                            ]
                        })
            except Exception as e:
                pass

            # 3. Datasets
            try:
                resp = requests.get(f"{base_url}/datasets", headers=headers, timeout=15)
                if resp.status_code == 200:
                    datasets = resp.json().get("datasets", [])
                    loc_datasets_count = len(datasets)
                    for ds in datasets:
                        meta = ds.get("metadata", {})
                        bq_source = meta.get("inputConfig", {}).get("bigquerySource", {}).get("uri")
                        datasets_meta.append({
                            "project_id": self.project_id,
                            "name": ds.get("name"),
                            "display_name": ds.get("displayName"),
                            "description": ds.get("description", ""),
                            "metadata_schema_uri": ds.get("metadataSchemaUri"),
                            "create_time": ds.get("createTime"),
                            "update_time": ds.get("updateTime"),
                            "location": loc,
                            "linked_bigquery_source": bq_source,
                            "model_reference": ds.get("modelReference"),
                            "prompt_type": meta.get("promptType")
                        })
            except Exception as e:
                pass

            # 4. Pipeline Jobs
            try:
                resp = requests.get(f"{base_url}/pipelineJobs", headers=headers, timeout=15)
                if resp.status_code == 200:
                    pipelines = resp.json().get("pipelineJobs", [])
                    for p in pipelines:
                        pipelines_meta.append({
                            "project_id": self.project_id,
                            "name": p.get("name"),
                            "display_name": p.get("displayName"),
                            "state": p.get("state"),
                            "create_time": p.get("createTime"),
                            "start_time": p.get("startTime"),
                            "end_time": p.get("endTime"),
                            "location": loc,
                            "template_uri": p.get("templateUri")
                        })
            except Exception as e:
                pass

            print(f"[{self.project_id}]    ⏳ [{loc}] Vertex AI -> {loc_models_count} modelos, {loc_datasets_count} datasets", flush=True)

        print(f"[{self.project_id}] ✅ [2/8 Vertex AI] Extraídos {len(models_meta)} modelos, {len(endpoints_meta)} endpoints, "
              f"{len(datasets_meta)} datasets e {len(pipelines_meta)} pipeline jobs.", flush=True)
        return {
            "models": models_meta,
            "endpoints": endpoints_meta,
            "datasets": datasets_meta,
            "pipeline_jobs": pipelines_meta
        }


class ComposerAirflowExtractor:
    """
    Extrator de metadados do Cloud Composer / Apache Airflow:
    - Ambientes gerenciados (Composer 2/3, Airflow 2.x)
    - Versão de imagem, cluster GKE subjacente, sizing de scheduler/worker
    - Bucket GCS de DAGs
    - Inventário de pacotes PyPI instalados
    - Variáveis de ambiente (sanitizadas com proteção LGPD)
    """
    def __init__(self, auth: GCPAuthManager, project_id: str, locations: Optional[List[str]] = None):
        self.auth = auth
        self.project_id = project_id
        self.locations = locations or ["us-central1", "southamerica-east1", "us-east1", "us-west1"]

    def extract_all(self) -> Dict[str, Any]:
        print(f"[{self.project_id}] 🌪️ [3/8 Composer/Airflow] Iniciando varredura...", flush=True)
        environments_meta = []
        headers = self.auth.get_headers()

        for loc in self.locations:
            url = f"https://composer.googleapis.com/v1/projects/{self.project_id}/locations/{loc}/environments"
            loc_envs = 0
            try:
                resp = requests.get(url, headers=headers, timeout=15)
                if resp.status_code == 200:
                    envs = resp.json().get("environments", [])
                    loc_envs = len(envs)
                    for e in envs:
                        cfg = e.get("config", {})
                        soft_cfg = cfg.get("softwareConfig", {})
                        node_cfg = cfg.get("nodeConfig", {})
                        workloads_cfg = cfg.get("workloadsConfig", {})

                        # Sanitização estrita de variáveis de ambiente
                        raw_env_vars = soft_cfg.get("envVariables", {})
                        safe_env_vars = LGPDGuardrail.sanitize_env_vars(raw_env_vars)

                        environments_meta.append({
                            "project_id": self.project_id,
                            "name": e.get("name"),
                            "display_name": e.get("name", "").split("/")[-1],
                            "state": e.get("state"),
                            "location": loc,
                            "create_time": e.get("createTime"),
                            "update_time": e.get("updateTime"),
                            "airflow_uri": cfg.get("airflowUri"),
                            "dag_gcs_prefix": cfg.get("dagGcsPrefix"),
                            "image_version": soft_cfg.get("imageVersion"),
                            "pypi_packages": list(soft_cfg.get("pypiPackages", {}).keys()),
                            "sanitized_env_variables_count": len(safe_env_vars),
                            "sanitized_env_variable_keys": list(safe_env_vars.keys()),
                            "node_machine_type": node_cfg.get("machineType"),
                            "scheduler_cpu": workloads_cfg.get("scheduler", {}).get("cpu"),
                            "worker_cpu": workloads_cfg.get("worker", {}).get("cpu"),
                            "triggerer_cpu": workloads_cfg.get("triggerer", {}).get("cpu")
                        })
            except Exception as e:
                pass
            print(f"[{self.project_id}]    ⏳ [{loc}] Cloud Composer -> {loc_envs} ambientes", flush=True)

        print(f"[{self.project_id}] ✅ [3/8 Composer/Airflow] Concluído: {len(environments_meta)} ambientes Cloud Composer.", flush=True)
        return {
            "environments": environments_meta
        }


class DataAgentsExtractor:
    """
    Extrator de Agentes Conversacionais de Dados (BigQuery Data Agents / Gemini Data Analytics).
    Captura:
    - Display Name e ID
    - System Instructions (Personas executivas de negócio)
    - Example / Verified Queries (Perguntas de Ouro validadas)
    - Fontes de conhecimento (Tabelas BigQuery e Property Graphs vinculados)
    """
    def __init__(self, auth: GCPAuthManager, project_id: str, locations: Optional[List[str]] = None):
        self.auth = auth
        self.project_id = project_id
        self.locations = locations or ["global", "us-central1"]

    def extract_all(self) -> Dict[str, Any]:
        print(f"[{self.project_id}] 💬 [4/8 Data Agents] Iniciando varredura...", flush=True)
        agents_meta = []
        headers = self.auth.get_headers()

        for loc in self.locations:
            url = f"https://geminidataanalytics.googleapis.com/v1alpha/projects/{self.project_id}/locations/{loc}/dataAgents"
            loc_agents = 0
            try:
                resp = requests.get(url, headers=headers, timeout=20)
                if resp.status_code == 200:
                    agents = resp.json().get("dataAgents", [])
                    loc_agents = len(agents)
                    for a in agents:
                        agent_id = a.get("name", "").split("/")[-1]
                        analytics_agent = a.get("dataAnalyticsAgent", {})
                        pub_ctx = analytics_agent.get("publishedContext") or analytics_agent.get("stagingContext") or {}
                        
                        system_instruction = pub_ctx.get("systemInstruction", "")
                        example_queries = pub_ctx.get("exampleQueries", [])
                        ds_refs = pub_ctx.get("datasourceReferences", {}).get("bq", {})
                        
                        table_refs = [
                            f"{t.get('projectId')}.{t.get('datasetId')}.{t.get('tableId')}"
                            for t in ds_refs.get("tableReferences", [])
                        ]
                        graph_refs = [
                            f"{g.get('projectId')}.{g.get('datasetId')}.{g.get('propertyGraphId')}"
                            for g in ds_refs.get("propertyGraphReferences", [])
                        ]

                        agents_meta.append({
                            "project_id": self.project_id,
                            "agent_id": agent_id,
                            "resource_name": a.get("name"),
                            "display_name": a.get("displayName"),
                            "description": a.get("description", ""),
                            "location": loc,
                            "create_time": a.get("createTime"),
                            "update_time": a.get("updateTime"),
                            "system_instruction_preview": (system_instruction[:300] + "...") if len(system_instruction) > 300 else system_instruction,
                            "full_system_instruction": system_instruction,
                            "verified_queries_count": len(example_queries),
                            "verified_queries": [
                                {
                                    "question": q.get("naturalLanguageQuestion"),
                                    "sql": q.get("sqlQuery")
                                }
                                for q in example_queries
                            ],
                            "referenced_tables": table_refs,
                            "referenced_property_graphs": graph_refs
                        })
            except Exception as e:
                pass
            print(f"[{self.project_id}]    ⏳ [{loc}] Data Agents -> {loc_agents} agentes", flush=True)

        print(f"[{self.project_id}] ✅ [4/8 Data Agents] Concluído: {len(agents_meta)} agentes conversacionais mapeados.", flush=True)
        return {
            "data_agents": agents_meta
        }


class DataplexCatalogExtractor:
    """
    Extrator de metadados de Governança do Knowledge Catalog & Dataplex:
    - Scans de Perfil de Dados (DATA_PROFILE)
    - Scans de Documentação (DATA_DOCUMENTATION)
    - Scans de Qualidade de Dados (DATA_QUALITY)
    - Status de execução e labels de publicação
    """
    def __init__(self, auth: GCPAuthManager, project_id: str, locations: Optional[List[str]] = None):
        self.auth = auth
        self.project_id = project_id
        self.locations = locations or ["us-central1", "global"]

    def extract_all(self) -> Dict[str, Any]:
        print(f"[{self.project_id}] 🛡️ [5/8 Dataplex/Catalog] Iniciando varredura...", flush=True)
        scans_meta = []
        headers = self.auth.get_headers()

        for loc in self.locations:
            url = f"https://dataplex.googleapis.com/v1/projects/{self.project_id}/locations/{loc}/dataScans"
            next_token = None
            page_count = 0
            while True:
                page_count += 1
                params = {}
                if next_token:
                    params["pageToken"] = next_token
                try:
                    resp = requests.get(url, headers=headers, params=params, timeout=20)
                    if resp.status_code == 200:
                        data = resp.json()
                        data_scans = data.get("dataScans", [])
                        for s in data_scans:
                            target_resource = s.get("data", {}).get("resource", "")
                            exec_status = s.get("executionStatus", {})
                            scans_meta.append({
                                "project_id": self.project_id,
                                "scan_id": s.get("name", "").split("/")[-1],
                                "display_name": s.get("displayName") or s.get("name", "").split("/")[-1],
                                "description": s.get("description", ""),
                                "type": s.get("type"),
                                "state": s.get("state"),
                                "target_resource": target_resource,
                                "create_time": s.get("createTime"),
                                "update_time": s.get("updateTime"),
                                "latest_job_start_time": exec_status.get("latestJobStartTime"),
                                "latest_job_end_time": exec_status.get("latestJobEndTime"),
                                "location": loc
                            })
                        if len(data_scans) > 0 and page_count % 3 == 0:
                            print(f"[{self.project_id}]    ⏳ [{loc}] Dataplex: Lote {page_count}, {len(scans_meta)} scans encontrados até agora...", flush=True)

                        next_token = data.get("nextPageToken")
                        if not next_token or page_count >= 10:  # Limite de segurança de paginação
                            break
                    else:
                        break
                except Exception as e:
                    break

        print(f"[{self.project_id}] ✅ [5/8 Dataplex/Catalog] Concluído: {len(scans_meta)} scans de governança/perfilamento.", flush=True)
        return {
            "dataplex_scans": scans_meta
        }


class LineagePipelinesExtractor:
    """
    Extrator de Linhagem de Dados e Pipelines de Carga:
    - Data Lineage API (processos e links de transformação upstream/downstream)
    - Cloud Dataflow (Jobs de streaming e batch)
    - Cloud Datastream (Streams de captura CDC e perfis de conexão)
    - Cloud Dataproc (Clusters Spark/Hadoop e Batches Serverless)
    """
    def __init__(self, auth: GCPAuthManager, project_id: str, locations: Optional[List[str]] = None):
        self.auth = auth
        self.project_id = project_id
        self.locations = locations or ["us-central1"]

    def extract_all(self, sample_tables_for_lineage: Optional[List[str]] = None) -> Dict[str, Any]:
        print(f"[{self.project_id}] 🔗 [6/8 Linhagem & Pipelines] Iniciando varredura...", flush=True)
        lineage_processes = []
        lineage_links = []
        dataflow_jobs = []
        datastream_streams = []
        dataproc_clusters = []
        dataproc_batches = []

        headers = self.auth.get_headers()

        for loc in self.locations:
            # 1. Data Lineage API: Processos
            url_proc = f"https://datalineage.googleapis.com/v1/projects/{self.project_id}/locations/{loc}/processes"
            try:
                resp = requests.get(url_proc, headers=headers, timeout=15)
                if resp.status_code == 200:
                    procs = resp.json().get("processes", [])
                    for p in procs[:50]:  # Top 50 processos mais recentes
                        lineage_processes.append({
                            "project_id": self.project_id,
                            "process_id": p.get("name", "").split("/")[-1],
                            "display_name": p.get("displayName"),
                            "origin_source_type": p.get("origin", {}).get("sourceType"),
                            "bigquery_job_id": p.get("attributes", {}).get("bigquery_job_id"),
                            "location": loc
                        })
            except Exception as e:
                pass

            # 2. Data Lineage API: Links de Linhagem para Tabelas de Amostra
            if sample_tables_for_lineage:
                url_links = f"https://datalineage.googleapis.com/v1/projects/{self.project_id}/locations/{loc}:searchLinks"
                for target_fqn in sample_tables_for_lineage[:20]:
                    try:
                        payload = {"target": {"fullyQualifiedName": f"bigquery:{target_fqn}"}}
                        resp = requests.post(url_links, headers=headers, json=payload, timeout=10)
                        if resp.status_code == 200:
                            links = resp.json().get("links", [])
                            for l in links:
                                lineage_links.append({
                                    "project_id": self.project_id,
                                    "target": l.get("target", {}).get("fullyQualifiedName"),
                                    "source": l.get("source", {}).get("fullyQualifiedName"),
                                    "start_time": l.get("startTime"),
                                    "end_time": l.get("endTime")
                                })
                    except Exception:
                        pass

            # 3. Cloud Dataflow Jobs
            url_df = f"https://dataflow.googleapis.com/v1b3/projects/{self.project_id}/locations/{loc}/jobs"
            try:
                resp = requests.get(url_df, headers=headers, timeout=15)
                if resp.status_code == 200:
                    jobs = resp.json().get("jobs", [])
                    for j in jobs[:50]:
                        dataflow_jobs.append({
                            "project_id": self.project_id,
                            "id": j.get("id"),
                            "name": j.get("name"),
                            "type": j.get("type"),
                            "current_state": j.get("currentState"),
                            "create_time": j.get("createTime"),
                            "location": loc
                        })
            except Exception as e:
                pass

            # 4. Cloud Datastream Streams
            url_ds = f"https://datastream.googleapis.com/v1/projects/{self.project_id}/locations/{loc}/streams"
            try:
                resp = requests.get(url_ds, headers=headers, timeout=15)
                if resp.status_code == 200:
                    streams = resp.json().get("streams", [])
                    for s in streams:
                        datastream_streams.append({
                            "project_id": self.project_id,
                            "stream_id": s.get("name", "").split("/")[-1],
                            "display_name": s.get("displayName"),
                            "state": s.get("state"),
                            "source_connection_profile": s.get("sourceConfig", {}).get("sourceConnectionProfile"),
                            "destination_connection_profile": s.get("destinationConfig", {}).get("destinationConnectionProfile"),
                            "location": loc
                        })
            except Exception as e:
                pass

            # 5. Cloud Dataproc Clusters & Serverless Batches
            url_dp_clusters = f"https://dataproc.googleapis.com/v1/projects/{self.project_id}/regions/{loc}/clusters"
            try:
                resp = requests.get(url_dp_clusters, headers=headers, timeout=15)
                if resp.status_code == 200:
                    clusters = resp.json().get("clusters", [])
                    for c in clusters:
                        status = c.get("status", {})
                        dataproc_clusters.append({
                            "project_id": self.project_id,
                            "cluster_name": c.get("clusterName"),
                            "status": status.get("state"),
                            "create_time": status.get("stateStartTime"),
                            "region": loc
                        })
            except Exception as e:
                pass

            url_dp_batches = f"https://dataproc.googleapis.com/v1/projects/{self.project_id}/locations/{loc}/batches"
            try:
                resp = requests.get(url_dp_batches, headers=headers, timeout=15)
                if resp.status_code == 200:
                    batches = resp.json().get("batches", [])
                    for b in batches[:30]:
                        dataproc_batches.append({
                            "project_id": self.project_id,
                            "batch_id": b.get("name", "").split("/")[-1],
                            "state": b.get("state"),
                            "create_time": b.get("createTime"),
                            "location": loc
                        })
            except Exception as e:
                pass

            print(f"[{self.project_id}]    ⏳ [{loc}] Linhagem/Pipelines -> {len(lineage_processes)} processos, {len(dataflow_jobs)} jobs", flush=True)

        print(f"[{self.project_id}] ✅ [6/8 Linhagem & Pipelines] Concluído: {len(lineage_processes)} processos, {len(lineage_links)} links, "
              f"{len(dataflow_jobs)} jobs Dataflow, {len(datastream_streams)} streams e {len(dataproc_clusters)} clusters Dataproc.", flush=True)
        return {
            "lineage_processes": lineage_processes,
            "lineage_links": lineage_links,
            "dataflow_jobs": dataflow_jobs,
            "datastream_streams": datastream_streams,
            "dataproc_clusters": dataproc_clusters,
            "dataproc_batches": dataproc_batches
        }


class DatabasesExtractor:
    """
    Extrator de metadados de Bancos Operacionais (Cloud SQL & Cloud Spanner):
    - Instâncias Cloud SQL, engines (Postgres, MySQL, SQL Server), bancos e flags
    - Instâncias Spanner, bancos, DDLs completas (tabelas, índices vetoriais, streams e grafos)
    """
    def __init__(self, auth: GCPAuthManager, project_id: str):
        self.auth = auth
        self.project_id = project_id

    def extract_all(self) -> Dict[str, Any]:
        print(f"[{self.project_id}] 🗄️ [7/8 Bancos Operacionais] Iniciando varredura...", flush=True)
        cloud_sql_instances = []
        spanner_instances = []
        headers = self.auth.get_headers()

        # 1. Cloud SQL
        url_sql = f"https://sqladmin.googleapis.com/v1/projects/{self.project_id}/instances"
        try:
            resp = requests.get(url_sql, headers=headers, timeout=15)
            if resp.status_code == 200:
                instances = resp.json().get("items", [])
                for inst in instances:
                    inst_name = inst.get("name")
                    settings = inst.get("settings", {})
                    
                    # Buscar bancos da instância
                    databases_list = []
                    url_dbs = f"https://sqladmin.googleapis.com/v1/projects/{self.project_id}/instances/{inst_name}/databases"
                    try:
                        resp_dbs = requests.get(url_dbs, headers=headers, timeout=10)
                        if resp_dbs.status_code == 200:
                            for d in resp_dbs.json().get("items", []):
                                databases_list.append({
                                    "name": d.get("name"),
                                    "charset": d.get("charset"),
                                    "collation": d.get("collation")
                                })
                    except Exception:
                        pass

                    cloud_sql_instances.append({
                        "project_id": self.project_id,
                        "instance_name": inst_name,
                        "database_version": inst.get("databaseVersion"),
                        "state": inst.get("state"),
                        "region": inst.get("region"),
                        "tier": settings.get("tier"),
                        "data_disk_size_gb": settings.get("dataDiskSizeGb"),
                        "storage_auto_resize": settings.get("storageAutoResize"),
                        "databases": databases_list
                    })
        except Exception as e:
            print(f"[{self.project_id}] ⚠️ [Cloud SQL] Erro ao listar instâncias: {e}", flush=True)

        # 2. Cloud Spanner
        url_spanner = f"https://spanner.googleapis.com/v1/projects/{self.project_id}/instances"
        try:
            resp = requests.get(url_spanner, headers=headers, timeout=15)
            if resp.status_code == 200:
                instances = resp.json().get("instances", [])
                for inst in instances:
                    inst_name = inst.get("name", "").split("/")[-1]
                    spanner_dbs = []

                    # Listar bancos desta instância
                    url_spanner_dbs = f"https://spanner.googleapis.com/v1/projects/{self.project_id}/instances/{inst_name}/databases"
                    try:
                        resp_sdbs = requests.get(url_spanner_dbs, headers=headers, timeout=15)
                        if resp_sdbs.status_code == 200:
                            for d in resp_sdbs.json().get("databases", []):
                                db_name = d.get("name", "").split("/")[-1]
                                
                                # Consultar DDL completa do banco Spanner
                                url_ddl = f"https://spanner.googleapis.com/v1/projects/{self.project_id}/instances/{inst_name}/databases/{db_name}/ddl"
                                ddl_statements = []
                                try:
                                    resp_ddl = requests.get(url_ddl, headers=headers, timeout=15)
                                    if resp_ddl.status_code == 200:
                                        ddl_statements = resp_ddl.json().get("statements", [])
                                except Exception:
                                    pass

                                # Análise sintética do DDL (tabelas, índices, grafos)
                                tables_count = sum(1 for s in ddl_statements if s.strip().startswith("CREATE TABLE"))
                                vector_indexes = [s for s in ddl_statements if "CREATE VECTOR INDEX" in s]
                                property_graphs = [s for s in ddl_statements if "CREATE PROPERTY GRAPH" in s or "CREATE OR REPLACE PROPERTY GRAPH" in s]

                                spanner_dbs.append({
                                    "database_name": db_name,
                                    "dialect": d.get("databaseDialect"),
                                    "state": d.get("state"),
                                    "tables_count": tables_count,
                                    "vector_indexes_count": len(vector_indexes),
                                    "property_graphs_count": len(property_graphs),
                                    "ddl_statements_count": len(ddl_statements),
                                    "ddl_statements": ddl_statements
                                })
                    except Exception:
                        pass

                    spanner_instances.append({
                        "project_id": self.project_id,
                        "instance_name": inst_name,
                        "display_name": inst.get("displayName"),
                        "edition": inst.get("edition"),
                        "node_count": inst.get("nodeCount"),
                        "processing_units": inst.get("processingUnits"),
                        "state": inst.get("state"),
                        "databases": spanner_dbs
                    })
        except Exception as e:
            print(f"[{self.project_id}] ⚠️ [Cloud Spanner] Erro ao listar instâncias: {e}", flush=True)

        print(f"[{self.project_id}] ✅ [Databases] Extraídas {len(cloud_sql_instances)} instâncias Cloud SQL e {len(spanner_instances)} instâncias Spanner.", flush=True)
        return {
            "cloud_sql": cloud_sql_instances,
            "spanner": spanner_instances
        }


class GCPOrganizationDiscovery:
    """
    Descobridor de Projetos Corporativos GCP em nível de Organização, Pasta ou Conta.
    Utiliza a Cloud Resource Manager API v3 (projects:search) com fallback para gcloud CLI.
    """
    def __init__(self, auth: GCPAuthManager):
        self.auth = auth

    def list_active_projects(
        self,
        organization_id: Optional[str] = None,
        folder_id: Optional[str] = None,
        project_filter: Optional[List[str]] = None
    ) -> List[Dict[str, Any]]:
        headers = self.auth.get_headers()
        projects = []

        # 1. Busca via Cloud Resource Manager API v3 (projects:search)
        try:
            query = "state:ACTIVE"
            url = f"https://cloudresourcemanager.googleapis.com/v3/projects:search?query={query}"
            next_token = None
            while True:
                params = {"pageToken": next_token} if next_token else {}
                resp = requests.get(url, headers=headers, params=params, timeout=20)
                if resp.status_code == 200:
                    data = resp.json()
                    for p in data.get("projects", []):
                        pid = p.get("projectId")
                        if not pid:
                            continue
                        parent = p.get("parent", "")
                        projects.append({
                            "project_id": pid,
                            "display_name": p.get("displayName") or pid,
                            "parent": parent,
                            "state": p.get("state"),
                            "create_time": p.get("createTime")
                        })
                    next_token = data.get("nextPageToken")
                    if not next_token:
                        break
                else:
                    break
        except Exception:
            pass

        # 2. Fallback via gcloud CLI se a API direta falhar
        if not projects:
            try:
                import subprocess, json
                cmd = ["gcloud", "projects", "list", "--filter=lifecycleState:ACTIVE", "--format=json"]
                out = subprocess.check_output(cmd, timeout=30).decode("utf-8")
                for p in json.loads(out):
                    projects.append({
                        "project_id": p.get("projectId"),
                        "display_name": p.get("name", p.get("projectId")),
                        "parent": str(p.get("parent", {})),
                        "state": p.get("lifecycleState"),
                        "create_time": p.get("createTime")
                    })
            except Exception:
                pass

        # 3. Aplicar Filtros Opcionais de Organização e Pasta
        if organization_id and organization_id.strip():
            org_num = organization_id.replace("organizations/", "").strip()
            filtered = [p for p in projects if org_num in str(p.get("parent", ""))]
            if filtered:
                projects = filtered

        if folder_id and folder_id.strip():
            folder_num = folder_id.replace("folders/", "").strip()
            filtered = [p for p in projects if folder_num in str(p.get("parent", ""))]
            if filtered:
                projects = filtered

        if project_filter:
            cleaned_filter = set([p.strip() for p in project_filter if p.strip()])
            if cleaned_filter:
                projects = [p for p in projects if p["project_id"] in cleaned_filter]

        # Desduplicação mantendo ordem
        seen = set()
        unique_projects = []
        for p in projects:
            if p["project_id"] not in seen:
                seen.add(p["project_id"])
                unique_projects.append(p)

        return unique_projects


class GCPEnterpriseAssessmentOrchestrator:
    """
    Orquestrador Central de Assessment Corporativo Google Cloud.
    Suporta varredura em Nível de Organização (Multi-Projeto), Pastas ou Projeto Único.
    Executa a extração em todas as 8 camadas, calcula o Scorecard de Governança
    e gera as saídas padronizadas:
    - metadata_assessment_manifest.json (JSON canônico estruturado com 'project_id' em todas as entidades)
    - data_catalog_dictionary.csv (Dicionário tabular de colunas, com project_id como 1ª coluna)
    - executive_assessment_summary.md (Relatório Executivo C-Level com Breakdown por Projeto)
    - metadata_assessment_<scope>.zip (Pacote ZIP consolidado)
    """
    def __init__(
        self,
        project_id: Optional[str] = None,
        project_ids: Optional[List[str]] = None,
        organization_id: Optional[str] = None,
        folder_id: Optional[str] = None,
        scope: str = "AUTO",
        output_dir: str = "output",
        gcs_output_uri: Optional[str] = None,
        max_workers: int = 4,
        enable_business_assessment: bool = True
    ):
        self.auth = GCPAuthManager(project_id=project_id)
        self.host_project_id = project_id or (project_ids[0] if project_ids else None) or self.auth.project_id
        self.scope = scope.upper() if scope else "AUTO"
        self.organization_id = organization_id
        self.folder_id = folder_id
        self.project_ids = project_ids or ([project_id] if project_id else [])
        self.output_dir = output_dir
        self.gcs_output_uri = gcs_output_uri
        self.max_workers = max(1, int(max_workers))
        self.enable_business_assessment = bool(enable_business_assessment)
        os.makedirs(self.output_dir, exist_ok=True)
        self.discovery = GCPOrganizationDiscovery(self.auth)

    def _audit_single_project(
        self,
        p_meta: Dict[str, Any],
        proj_idx: int,
        total_projs: int,
        dataset_filter: Optional[List[str]] = None
    ) -> Dict[str, Any]:
        """Executa a auditoria das 8 camadas para um único projeto GCP."""
        pid = p_meta["project_id"]
        pname = p_meta.get("display_name", pid)
        p_start = time.time()
        print(f"\n🚀 [{pid}] [INÍCIO {proj_idx}/{total_projs}] Auditoria iniciada ('{pname}')", flush=True)

        bq_ext = BigQueryMetadataExtractor(self.auth, pid)
        vertex_ext = VertexAIExtractor(self.auth, pid)
        composer_ext = ComposerAirflowExtractor(self.auth, pid)
        agents_ext = DataAgentsExtractor(self.auth, pid)
        dataplex_ext = DataplexCatalogExtractor(self.auth, pid)
        lineage_ext = LineagePipelinesExtractor(self.auth, pid)
        db_ext = DatabasesExtractor(self.auth, pid)

        # 1. BigQuery
        try:
            bq_data = bq_ext.extract_all(dataset_filter=dataset_filter)
        except Exception as e:
            print(f"[{pid}] ⚠️ [BigQuery] Falha ao extrair: {e}", flush=True)
            bq_data = {"datasets": [], "tables_and_views": [], "property_graphs": [], "ml_models": []}

        # 2. Vertex AI
        try:
            vertex_data = vertex_ext.extract_all()
        except Exception as e:
            print(f"[{pid}] ⚠️ [Vertex AI] Falha ao extrair: {e}", flush=True)
            vertex_data = {"models": [], "endpoints": [], "datasets": [], "pipeline_jobs": []}

        # 3. Cloud Composer & Airflow
        try:
            composer_data = composer_ext.extract_all()
        except Exception as e:
            print(f"[{pid}] ⚠️ [Composer] Falha ao extrair: {e}", flush=True)
            composer_data = {"environments": []}

        # 4. Data Agents
        try:
            agents_data = agents_ext.extract_all()
        except Exception as e:
            print(f"[{pid}] ⚠️ [Data Agents] Falha ao extrair: {e}", flush=True)
            agents_data = {"data_agents": []}

        # 5. Dataplex / Knowledge Catalog
        try:
            dataplex_data = dataplex_ext.extract_all()
        except Exception as e:
            print(f"[{pid}] ⚠️ [Dataplex] Falha ao extrair: {e}", flush=True)
            dataplex_data = {"dataplex_scans": []}

        # 6. Linhagem e Pipelines
        try:
            sample_tables = [
                f"{t['project_id']}.{t['dataset_id']}.{t['table_name']}"
                for t in bq_data.get("tables_and_views", [])
                if t.get("table_type") in ("VIEW", "MATERIALIZED VIEW")
            ][:15]
            lineage_data = lineage_ext.extract_all(sample_tables_for_lineage=sample_tables)
        except Exception as e:
            print(f"[{pid}] ⚠️ [Linhagem] Falha ao extrair: {e}", flush=True)
            lineage_data = {
                "lineage_processes": [], "lineage_links": [], "dataflow_jobs": [],
                "datastream_streams": [], "dataproc_clusters": [], "dataproc_batches": []
            }

        # 7. Bancos Operacionais
        try:
            db_data = db_ext.extract_all()
        except Exception as e:
            print(f"[{pid}] ⚠️ [Bancos] Falha ao extrair: {e}", flush=True)
            db_data = {"cloud_sql": [], "spanner": []}

        proj_tables = [t for t in bq_data.get("tables_and_views", []) if t.get("table_type") in ("BASE TABLE", "TABLE")]
        proj_views = [t for t in bq_data.get("tables_and_views", []) if t.get("table_type") in ("VIEW", "MATERIALIZED VIEW")]
        proj_cols = sum(t.get("columns_count", 0) for t in bq_data.get("tables_and_views", []))
        proj_doc_cols = sum(t.get("documented_columns_count", 0) for t in bq_data.get("tables_and_views", []))
        proj_doc_pct = round((proj_doc_cols / proj_cols * 100), 2) if proj_cols > 0 else 0.0
        p_elapsed = round(time.time() - p_start, 2)

        print(f"🏁 [{pid}] [CONCLUÍDO {proj_idx}/{total_projs}] Finalizado em {p_elapsed}s | "
              f"{len(bq_data.get('datasets', []))} datasets, {len(proj_tables)+len(proj_views)} tabelas/views, "
              f"{len(agents_data.get('data_agents', []))} agents, {len(vertex_data.get('models', []))} vertex models", flush=True)

        summary = {
            "project_id": pid,
            "display_name": pname,
            "parent": p_meta.get("parent", "N/A"),
            "duration_seconds": p_elapsed,
            "datasets_count": len(bq_data.get("datasets", [])),
            "tables_count": len(proj_tables),
            "views_count": len(proj_views),
            "columns_count": proj_cols,
            "documented_columns_count": proj_doc_cols,
            "documentation_coverage_pct": proj_doc_pct,
            "property_graphs_count": len(bq_data.get("property_graphs", [])),
            "bqml_models_count": len(bq_data.get("ml_models", [])),
            "vertex_models_count": len(vertex_data.get("models", [])),
            "data_agents_count": len(agents_data.get("data_agents", [])),
            "dataplex_scans_count": len(dataplex_data.get("dataplex_scans", [])),
            "composer_environments_count": len(composer_data.get("environments", [])),
            "dataflow_jobs_count": len(lineage_data.get("dataflow_jobs", [])),
            "datastream_streams_count": len(lineage_data.get("datastream_streams", [])),
            "cloud_sql_count": len(db_data.get("cloud_sql", [])),
            "spanner_count": len(db_data.get("spanner", []))
        }

        return {
            "project_summary": summary,
            "bq_data": bq_data,
            "vertex_data": vertex_data,
            "composer_data": composer_data,
            "agents_data": agents_data,
            "dataplex_data": dataplex_data,
            "lineage_data": lineage_data,
            "db_data": db_data
        }

    def run_assessment(self, dataset_filter: Optional[List[str]] = None) -> Dict[str, Any]:
        start_time = time.time()

        # 1. Determinação dos Projetos-Alvo
        if self.scope == "ORGANIZATION":
            target_projects = self.discovery.list_active_projects(
                organization_id=self.organization_id,
                folder_id=self.folder_id
            )
        elif self.scope == "CUSTOM_PROJECTS":
            target_projects = self.discovery.list_active_projects(project_filter=self.project_ids)
            if not target_projects and self.project_ids:
                target_projects = [{"project_id": pid, "display_name": pid} for pid in self.project_ids]
        elif self.scope == "CURRENT_PROJECT":
            target_projects = [{"project_id": self.host_project_id, "display_name": self.host_project_id}]
        else: # AUTO
            if self.organization_id:
                target_projects = self.discovery.list_active_projects(organization_id=self.organization_id)
            elif len(self.project_ids) > 1:
                target_projects = self.discovery.list_active_projects(project_filter=self.project_ids)
            else:
                target_projects = [{"project_id": self.host_project_id, "display_name": self.host_project_id}]

        if not target_projects:
            target_projects = [{"project_id": self.host_project_id, "display_name": self.host_project_id}]

        project_ids_list = [p["project_id"] for p in target_projects]
        effective_workers = min(self.max_workers, len(target_projects))

        print("\n" + "="*80, flush=True)
        print("🌐 GCP ENTERPRISE MULTI-PROJECT & ORGANIZATION METADATA ASSESSMENT", flush=True)
        print(f"🏢 Escopo: {self.scope} | Organização: {self.organization_id or 'Global/Auto-Detectada'}", flush=True)
        print(f"🎯 {len(target_projects)} Projetos Identificados: {', '.join(project_ids_list)}", flush=True)
        print(f"⚡ Paralelismo Concorrente: {effective_workers} workers simultâneos (lotes de {effective_workers} em {effective_workers})", flush=True)
        print("🔒 LGPD STATUS: MODO ESTRITO DE METADADOS (ZERO LEITURA DE DADOS DE CLIENTE)", flush=True)
        print("="*80 + "\n", flush=True)

        # Listas consolidadas
        all_datasets = []
        all_tables_and_views = []
        all_property_graphs = []
        all_ml_models = []
        all_vertex_models = []
        all_vertex_endpoints = []
        all_vertex_datasets = []
        all_vertex_pipelines = []
        all_composer_environments = []
        all_data_agents = []
        all_dataplex_scans = []
        all_lineage_processes = []
        all_lineage_links = []
        all_dataflow_jobs = []
        all_datastream_streams = []
        all_dataproc_clusters = []
        all_dataproc_batches = []
        all_cloud_sql_instances = []
        all_spanner_instances = []
        project_summaries = []

        # 2. Execução Paralela Projeto a Projeto
        print(f"🚀 Iniciando extração concorrente ({effective_workers} threads ativas)...", flush=True)
        results_by_project = {}
        with concurrent.futures.ThreadPoolExecutor(max_workers=effective_workers) as executor:
            future_to_pid = {
                executor.submit(self._audit_single_project, p_meta, idx, len(target_projects), dataset_filter): p_meta["project_id"]
                for idx, p_meta in enumerate(target_projects, 1)
            }
            for future in concurrent.futures.as_completed(future_to_pid):
                pid = future_to_pid[future]
                try:
                    res = future.result()
                    results_by_project[pid] = res
                except Exception as exc:
                    print(f"❌ [{pid}] Exceção não tratada durante a auditoria concorrente: {exc}", flush=True)

        # 3. Consolidação dos Resultados mantendo a ordem original dos projetos
        for p_meta in target_projects:
            pid = p_meta["project_id"]
            if pid not in results_by_project:
                continue
            res = results_by_project[pid]
            bq_data = res["bq_data"]
            vertex_data = res["vertex_data"]
            composer_data = res["composer_data"]
            agents_data = res["agents_data"]
            dataplex_data = res["dataplex_data"]
            lineage_data = res["lineage_data"]
            db_data = res["db_data"]

            all_datasets.extend(bq_data.get("datasets", []))
            all_tables_and_views.extend(bq_data.get("tables_and_views", []))
            all_property_graphs.extend(bq_data.get("property_graphs", []))
            all_ml_models.extend(bq_data.get("ml_models", []))
            all_vertex_models.extend(vertex_data.get("models", []))
            all_vertex_endpoints.extend(vertex_data.get("endpoints", []))
            all_vertex_datasets.extend(vertex_data.get("datasets", []))
            all_vertex_pipelines.extend(vertex_data.get("pipeline_jobs", []))
            all_composer_environments.extend(composer_data.get("environments", []))
            all_data_agents.extend(agents_data.get("data_agents", []))
            all_dataplex_scans.extend(dataplex_data.get("dataplex_scans", []))
            all_lineage_processes.extend(lineage_data.get("lineage_processes", []))
            all_lineage_links.extend(lineage_data.get("lineage_links", []))
            all_dataflow_jobs.extend(lineage_data.get("dataflow_jobs", []))
            all_datastream_streams.extend(lineage_data.get("datastream_streams", []))
            all_dataproc_clusters.extend(lineage_data.get("dataproc_clusters", []))
            all_dataproc_batches.extend(lineage_data.get("dataproc_batches", []))
            all_cloud_sql_instances.extend(db_data.get("cloud_sql", []))
            all_spanner_instances.extend(db_data.get("spanner", []))

            project_summaries.append(res["project_summary"])

        # 3. Cruzamento Global de Perfilamento Dataplex
        profiled_resources = {s.get("target_resource") for s in all_dataplex_scans if s.get("type") == "DATA_PROFILE"}
        doc_scans_resources = {s.get("target_resource") for s in all_dataplex_scans if s.get("type") == "DATA_DOCUMENTATION"}

        for t in all_tables_and_views:
            resource_uri = f"//bigquery.googleapis.com/projects/{t['project_id']}/datasets/{t['dataset_id']}/tables/{t['table_name']}"
            t["dataplex_profile_scan_active"] = resource_uri in profiled_resources
            t["dataplex_documentation_scan_active"] = resource_uri in doc_scans_resources

        # 4. Métricas Globais Consolidadas
        total_tables = len([t for t in all_tables_and_views if t.get("table_type") in ("BASE TABLE", "TABLE")])
        total_views = len([t for t in all_tables_and_views if t.get("table_type") in ("VIEW", "MATERIALIZED VIEW")])
        total_columns = sum(t.get("columns_count", 0) for t in all_tables_and_views)
        total_doc_cols = sum(t.get("documented_columns_count", 0) for t in all_tables_and_views)
        overall_doc_pct = round((total_doc_cols / total_columns * 100), 2) if total_columns > 0 else 0.0

        profiled_tables_count = sum(1 for t in all_tables_and_views if t.get("dataplex_profile_scan_active"))
        profile_coverage_pct = round((profiled_tables_count / len(all_tables_and_views) * 100), 2) if all_tables_and_views else 0.0

        total_bq_graphs = len(all_property_graphs)
        total_spanner_graphs = sum(
            db.get("property_graphs_count", 0)
            for inst in all_spanner_instances
            for db in inst.get("databases", [])
        )

        executive_summary = {
            "assessment_timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "scope": self.scope,
            "organization_id": self.organization_id,
            "host_project_id": self.host_project_id,
            "concurrency_mode": "PARALLEL",
            "max_workers": self.max_workers,
            "effective_workers": effective_workers,
            "total_projects_audited": len(target_projects),
            "projects_list": project_ids_list,
            "lgpd_compliance": "STRICT_METADATA_ONLY_ZERO_PII",
            "duration_seconds": round(time.time() - start_time, 2),
            "kpis": {
                "total_projects": len(target_projects),
                "total_datasets": len(all_datasets),
                "total_tables": total_tables,
                "total_views": total_views,
                "total_columns": total_columns,
                "documented_columns_count": total_doc_cols,
                "documentation_coverage_pct": overall_doc_pct,
                "profiled_tables_count": profiled_tables_count,
                "profile_coverage_pct": profile_coverage_pct,
                "total_property_graphs": total_bq_graphs + total_spanner_graphs,
                "total_bqml_models": len(all_ml_models),
                "total_vertex_models": len(all_vertex_models),
                "total_vertex_endpoints": len(all_vertex_endpoints),
                "total_data_agents": len(all_data_agents),
                "total_composer_environments": len(all_composer_environments),
                "total_dataflow_jobs": len(all_dataflow_jobs),
                "total_datastream_streams": len(all_datastream_streams),
                "total_dataproc_clusters": len(all_dataproc_clusters),
                "total_cloud_sql_instances": len(all_cloud_sql_instances),
                "total_spanner_instances": len(all_spanner_instances)
            }
        }

        # 5. Montagem do Manifesto Canônico Estruturado
        manifest = {
            "metadata_header": {
                "assessment_scope": self.scope,
                "organization_id": self.organization_id,
                "host_project_id": self.host_project_id,
                "concurrency_mode": "PARALLEL",
                "max_workers": self.max_workers,
                "effective_workers": effective_workers,
                "generated_at": executive_summary["assessment_timestamp"],
                "duration_seconds": executive_summary["duration_seconds"],
                "lgpd_guardrails": "STRICT_METADATA_ONLY_ZERO_PII",
                "total_projects_audited": len(target_projects),
                "projects_evaluated": project_summaries,
                "global_kpis": executive_summary["kpis"]
            },
            "bigquery": {
                "datasets": all_datasets,
                "tables_and_views": all_tables_and_views,
                "property_graphs": all_property_graphs,
                "ml_models": all_ml_models
            },
            "vertex_ai": {
                "models": all_vertex_models,
                "endpoints": all_vertex_endpoints,
                "datasets": all_vertex_datasets,
                "pipeline_jobs": all_vertex_pipelines
            },
            "composer_airflow": {
                "environments": all_composer_environments
            },
            "data_agents": {
                "data_agents": all_data_agents
            },
            "dataplex_catalog": {
                "dataplex_scans": all_dataplex_scans
            },
            "lineage_and_pipelines": {
                "lineage_processes": all_lineage_processes,
                "lineage_links": all_lineage_links,
                "dataflow_jobs": all_dataflow_jobs,
                "datastream_streams": all_datastream_streams,
                "dataproc_clusters": all_dataproc_clusters,
                "dataproc_batches": all_dataproc_batches
            },
            "operational_databases": {
                "cloud_sql": all_cloud_sql_instances,
                "spanner": all_spanner_instances
            }
        }

        # 6. Business Assessment: Geração de Casos de Negócio & Quick Wins
        if self.enable_business_assessment:
            try:
                try:
                    from business_assessment_engine import GCPBusinessAssessmentEngine
                except ImportError:
                    GCPBusinessAssessmentEngine = globals().get("GCPBusinessAssessmentEngine")
                
                if GCPBusinessAssessmentEngine:
                    print("\n💼 [Business Assessment] Formulando Casos de Negócio, Quick Wins & Matriz de Priorização...", flush=True)
                    biz_engine = GCPBusinessAssessmentEngine(manifest)
                    _, _, biz_data = biz_engine.export_artifacts(self.output_dir)
                    manifest["business_assessment"] = biz_data
            except Exception as e:
                print(f"⚠️ [Business Assessment] Aviso ao gerar casos de negócio: {e}", flush=True)

        # 7. Exportações Padronizadas
        print("\n📝 [Consolidação & Empacotamento] Gerando arquivos canônicos...", flush=True)
        self._export_manifest_json(manifest)
        self._export_catalog_dictionary_csv(all_tables_and_views)
        self._export_executive_summary_md(
            summary=executive_summary,
            bq_data={"property_graphs": all_property_graphs},
            agents_data={"data_agents": all_data_agents},
            vertex_data={"datasets": all_vertex_datasets, "pipeline_jobs": all_vertex_pipelines},
            composer_data={"environments": all_composer_environments},
            project_summaries=project_summaries
        )

        # 8. Criação do Pacote ZIP com Todos os Resultados
        zip_path = self._create_zip_package()

        # 9. Upload Automático para o GCS (se configurado)
        target_gcs = self.gcs_output_uri
        if target_gcs and target_gcs.startswith("gs://"):
            self._upload_to_gcs(target_gcs)

        print("\n" + "="*80, flush=True)
        print(f"🎉 ASSESSMENT MULTI-PROJETO CONCLUÍDO EM {executive_summary['duration_seconds']} SEGUNDOS!", flush=True)
        print(f"🏢 {len(target_projects)} Projetos Auditados com Sucesso (Paralelismo: {effective_workers} workers simultâneos)!", flush=True)
        print(f"📁 Arquivos gerados no diretório local: '{self.output_dir}'", flush=True)
        print(f"   🗜️ Pacote ZIP consolidado: '{zip_path}'", flush=True)
        print("   - metadata_assessment_manifest.json (Canônico com todas as entidades e project_id)", flush=True)
        print("   - data_catalog_dictionary.csv (Dicionário tabular de todas as colunas com project_id)", flush=True)
        print("   - executive_assessment_summary.md (Relatório Executivo C-Level com Breakdown por Projeto)", flush=True)
        if self.enable_business_assessment:
            print("   - business_assessment_cases.md (Relatório Estratégico de Casos de Negócio & Quick Wins)", flush=True)
            print("   - business_assessment_cases.json (Catálogo Estruturado de Casos de Negócio)", flush=True)
        if target_gcs and target_gcs.startswith("gs://"):
            print(f"☁️ Arquivos sincronizados no GCS: '{target_gcs}'", flush=True)
        print("="*80 + "\n", flush=True)

        return manifest

    def _create_zip_package(self) -> str:
        """Compacta todos os artefatos de saída em um arquivo ZIP único."""
        import zipfile
        scope_tag = "organization" if self.scope == "ORGANIZATION" else self.host_project_id
        zip_filename = f"metadata_assessment_{scope_tag}.zip"
        zip_filepath = os.path.join(self.output_dir, zip_filename)

        files_to_pack = [
            "metadata_assessment_manifest.json",
            "data_catalog_dictionary.csv",
            "executive_assessment_summary.md",
            "business_assessment_cases.md",
            "business_assessment_cases.json"
        ]

        print(f"🗜️ Compactando resultados em '{zip_filename}'...", end="", flush=True)
        with zipfile.ZipFile(zip_filepath, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
            for fname in files_to_pack:
                fpath = os.path.join(self.output_dir, fname)
                if os.path.exists(fpath):
                    zipf.write(fpath, arcname=fname)
        print(f" -> Concluído! ({round(os.path.getsize(zip_filepath)/1024, 2)} KB)", flush=True)
        return zip_filepath

    def _upload_to_gcs(self, gcs_uri: str) -> List[str]:
        """Faz upload de todos os artefatos gerados e do arquivo ZIP para a URI do Google Cloud Storage."""
        from google.cloud import storage
        print(f"\n☁️ [GCS Upload] Sincronizando arquivos e pacote ZIP com '{gcs_uri}'...", flush=True)
        clean_uri = gcs_uri.replace("gs://", "")
        parts = clean_uri.split("/", 1)
        bucket_name = parts[0]
        prefix = parts[1].rstrip("/") if len(parts) > 1 else ""

        try:
            storage_client = storage.Client(project=self.host_project_id, credentials=self.auth._credentials)
            bucket = storage_client.bucket(bucket_name)
            uploaded_blobs = []

            scope_tag = "organization" if self.scope == "ORGANIZATION" else self.host_project_id
            zip_filename = f"metadata_assessment_{scope_tag}.zip"
            files_to_upload = [
                zip_filename,
                "metadata_assessment_manifest.json",
                "data_catalog_dictionary.csv",
                "executive_assessment_summary.md"
            ]

            for filename in files_to_upload:
                local_path = os.path.join(self.output_dir, filename)
                if os.path.exists(local_path):
                    blob_name = f"{prefix}/{filename}" if prefix else filename
                    blob = bucket.blob(blob_name)
                    blob.upload_from_filename(local_path)
                    gcs_full_path = f"gs://{bucket_name}/{blob_name}"
                    uploaded_blobs.append(gcs_full_path)
                    print(f"   ⬆️ Upload concluído: {gcs_full_path}", flush=True)

            print(f"✅ [GCS Upload] {len(uploaded_blobs)} arquivos salvos com sucesso no bucket '{bucket_name}'.", flush=True)
            return uploaded_blobs
        except Exception as e:
            print(f"⚠️ [GCS Upload] Falha ao enviar arquivos para o GCS ({gcs_uri}): {e}", flush=True)
            return []

    def _export_manifest_json(self, manifest: Dict[str, Any]):
        filepath = os.path.join(self.output_dir, "metadata_assessment_manifest.json")
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(manifest, f, indent=2, ensure_ascii=False)
        print(f"📄 Manifesto JSON salvo em: {filepath}")

    def _export_catalog_dictionary_csv(self, tables_and_views: List[Dict[str, Any]]):
        rows = []
        for t in tables_and_views:
            for c in t.get("columns", []):
                rows.append({
                    "project_id": t.get("project_id"),
                    "dataset_id": t.get("dataset_id"),
                    "table_name": t.get("table_name"),
                    "table_type": t.get("table_type"),
                    "table_description": t.get("description", ""),
                    "column_name": c.get("column_name"),
                    "data_type": c.get("data_type"),
                    "column_description": c.get("description", ""),
                    "is_column_documented": bool(c.get("description", "").strip()),
                    "dataplex_profile_scan_active": t.get("dataplex_profile_scan_active", False),
                    "estimated_rows": t.get("num_rows_estimated", 0),
                    "estimated_bytes": t.get("total_bytes_estimated", 0)
                })

        df = pd.DataFrame(rows)
        filepath = os.path.join(self.output_dir, "data_catalog_dictionary.csv")
        df.to_csv(filepath, index=False, encoding="utf-8")
        print(f"📊 Dicionário de Dados CSV salvo em: {filepath} ({len(df)} colunas catalogadas)")

    def _export_executive_summary_md(
        self,
        summary: Dict[str, Any],
        bq_data: Dict[str, Any],
        agents_data: Dict[str, Any],
        vertex_data: Dict[str, Any],
        composer_data: Dict[str, Any],
        project_summaries: Optional[List[Dict[str, Any]]] = None
    ):
        k = summary["kpis"]
        filepath = os.path.join(self.output_dir, "executive_assessment_summary.md")
        
        md_content = f"""# Sumário Executivo: Assessment de Metadados & Maturidade GCP

**Escopo da Avaliação:** `{summary.get('scope', 'AUTO')}`  
**Modo de Execução:** `Paralelo ({summary.get('effective_workers', 4)} workers simultâneos)`  
**Organização / Raiz:** `{summary.get('organization_id') or 'Auto-Detectada'}`  
**Projetos Auditados:** `{summary.get('total_projects_audited', 1)}` ({', '.join(summary.get('projects_list', []))})  
**Data da Extração:** `{summary['assessment_timestamp']}`  
**Conformidade de Governança:** `LGPD Safe - Zero PII / Metadata-Only`  
**Tempo Total de Processamento:** `{summary['duration_seconds']}s`  

---

## 🏆 Scorecard de Governança & Prontidão para IA (Global)

| Dimensão de Avaliação | Métrica Observada | Meta de Mercado (Best Practice) | Status de Prontidão |
|---|---|---|---|
| **Documentação de Colunas (Grounding IA)** | **{k['documentation_coverage_pct']}%** ({k['documented_columns_count']}/{k['total_columns']}) | >= 80% | {"🟢 Alto" if k['documentation_coverage_pct'] >= 80 else ("🟡 Médio" if k['documentation_coverage_pct'] >= 50 else "🔴 Atenção Crítica")} |
| **Perfilamento no Knowledge Catalog** | **{k['profile_coverage_pct']}%** ({k['profiled_tables_count']}/{k['total_tables'] + k['total_views']}) | 100% | {"🟢 Conforme" if k['profile_coverage_pct'] >= 90 else "🟡 Requer Scans"} |
| **Grafos Semânticos (GQL)** | **{k['total_property_graphs']}** grafos registrados | >= 1 | {"🟢 Habilitado" if k['total_property_graphs'] > 0 else "⚪ Não Detectado"} |
| **Agentes Conversacionais (Data Agents)** | **{k['total_data_agents']}** agentes corporativos | Multi-domínio | {"🟢 Ativo" if k['total_data_agents'] > 0 else "⚪ Nenhum"} |
| **Modelos de IA & ML (BQML + Vertex)** | **{k['total_bqml_models'] + k['total_vertex_models']}** modelos | Unificados | {"🟢 Ativo" if (k['total_bqml_models'] + k['total_vertex_models']) > 0 else "⚪ Nenhum"} |

---
"""

        # Tabela de Breakdown por Projeto na Organização
        if project_summaries:
            md_content += """## 🏢 Inventário Consolidado por Projeto na Organização

| Projeto GCP | Datasets | Tabelas | Views | Colunas | % Doc | Data Agents | Modelos IA | Scans Dataplex | Composer |
|---|---|---|---|---|---|---|---|---|---|
"""
            for ps in project_summaries:
                md_content += f"| `{ps['project_id']}` | {ps['datasets_count']} | {ps['tables_count']} | {ps['views_count']} | {ps['columns_count']} | **{ps['documentation_coverage_pct']}%** | {ps['data_agents_count']} | {ps['vertex_models_count'] + ps['bqml_models_count']} | {ps['dataplex_scans_count']} | {ps['composer_environments_count']} |\n"
            md_content += "\n---\n\n"

        md_content += f"""## 📊 Inventário Consolidado de Ativos da Organização

### 1. Camada de Dados Analíticos (BigQuery)
- **Datasets Registrados:** `{k['total_datasets']}`
- **Tabelas de Base:** `{k['total_tables']}`
- **Views & Materialized Views:** `{k['total_views']}`
- **Property Graphs (GQL):** `{k['total_property_graphs']}`
- **Modelos BQML:** `{k['total_bqml_models']}`

### 2. Camada de Inteligência Artificial & MLOps (Vertex AI)
- **Vertex AI Model Registry:** `{k['total_vertex_models']}` modelos cadastrados
- **Endpoints de Inferência Ativos:** `{k['total_vertex_endpoints']}`
- **Datasets de IA (Treino/Prompt/Tabular):** `{len(vertex_data.get('datasets', []))}`
- **Pipelines de MLOps Registrados:** `{len(vertex_data.get('pipeline_jobs', []))}`

### 3. Orquestração & Engenharia de Dados (Composer / Airflow / Pipelines)
- **Ambientes Cloud Composer:** `{k['total_composer_environments']}`
- **Jobs do Cloud Dataflow:** `{k['total_dataflow_jobs']}`
- **Streams do Cloud Datastream (CDC):** `{k['total_datastream_streams']}`
- **Clusters Dataproc:** `{k['total_dataproc_clusters']}`

### 4. Camada de Bancos Operacionais
- **Instâncias Cloud SQL:** `{k['total_cloud_sql_instances']}`
- **Instâncias Cloud Spanner:** `{k['total_spanner_instances']}`

---

## 💬 Agentes Conversacionais (Data Agents) & Perguntas de Ouro

"""
        for a in agents_data.get("data_agents", []):
            md_content += f"""### Agente: {a['display_name']} (`{a['agent_id']}`) - Projeto: `{a.get('project_id', 'N/A')}`
- **Descrição:** {a.get('description', 'N/A')}
- **Localização:** `{a['location']}`
- **Fontes Vinculadas:** {', '.join(a['referenced_tables'] + a['referenced_property_graphs'])}
- **Perguntas Validadas ({a['verified_queries_count']}):**
"""
            for q in a.get("verified_queries", [])[:3]:
                md_content += f"  - *\"{q['question']}\"*\n"
            md_content += "\n"

        md_content += """---

## 🎯 Recomendações para Geração de Casos de Negócio

1. **Aceleração com Data Agents**: Aproveitar as tabelas que já possuem descrições de colunas para criar novos agentes executivos.
2. **Priorização de Profiling**: Rodar Data Profile Scans nas tabelas ainda não catalogadas no Knowledge Catalog para destravar o Gemini Reasoning Engine.
3. **Consolidação de Linhagem**: Conectar os pipelines do Composer e Dataflow com a Data Lineage API para rastreabilidade de ponta a ponta dos KPIs aos bancos operacionais.
"""

        with open(filepath, "w", encoding="utf-8") as f:
            f.write(md_content)
        print(f"📑 Sumário Executivo Markdown salvo em: {filepath}")

print('✅ Engine Corporativa de Extração, Business Assessment e LGPD Guardrails carregada com sucesso no ambiente!')


### 🚀 4. Disparo do Assessment Multi-Projeto & Sincronização com o GCS
Inicia a varredura automática dos projetos da organização e envia os artefatos consolidados para o GCS.

In [ ]:
# @title Iniciar Extração de Metadados Multi-Projeto
orchestrator = GCPEnterpriseAssessmentOrchestrator(
    organization_id=ORGANIZATION_ID.strip() if ORGANIZATION_ID.strip() else None,
    project_ids=PROJECT_IDS,
    scope=ASSESSMENT_SCOPE,
    output_dir=OUTPUT_DIR,
    gcs_output_uri=GCS_OUTPUT_URI.strip() if GCS_OUTPUT_URI.strip().startswith('gs://') else None,
    max_workers=MAX_WORKERS
)

manifest = orchestrator.run_assessment(dataset_filter=DATASET_FILTER)
kpis = manifest['metadata_header']['global_kpis']
projects_eval = manifest['metadata_header']['projects_evaluated']

print(f"\n🎉 Assessment Corporativo concluído com sucesso em {manifest['metadata_header']['duration_seconds']}s!")
print(f"🏢 Total de Projetos Auditados na Organização: {len(projects_eval)}")

### 🏢 5. Inventário Consolidado por Projeto na Organização
Tabela comparativa de ativos, maturidade e cobertura de governança entre todos os projetos auditados.

In [ ]:
# @title Tabela de Inventário por Projeto
df_proj_summary = pd.DataFrame(projects_eval)
cols_to_show = [
    'project_id', 'display_name', 'datasets_count', 'tables_count', 
    'views_count', 'columns_count', 'documentation_coverage_pct', 
    'data_agents_count', 'vertex_models_count', 'dataplex_scans_count', 'composer_environments_count'
]
existing_cols = [c for c in cols_to_show if c in df_proj_summary.columns]
display(HTML(df_proj_summary[existing_cols].to_html(classes='table table-striped table-hover', index=False)))

### 🏆 6. Scorecard Global de Governança & Prontidão para IA

In [ ]:
# @title Exibição do Scorecard Global de Governança
scorecard_data = [
    {
        "Dimensão": "Projetos Auditados",
        "Métrica Observada": f"{kpis['total_projects']} projetos",
        "Meta de Mercado": "100% da Organização",
        "Status": "🟢 Concluído"
    },
    {
        "Dimensão": "Documentação de Colunas (Grounding IA)",
        "Métrica Observada": f"{kpis['documentation_coverage_pct']}% ({kpis['documented_columns_count']}/{kpis['total_columns']})",
        "Meta de Mercado": ">= 80%",
        "Status": "🟢 Pronto para IA" if kpis['documentation_coverage_pct'] >= 80 else ("🟡 Parcial" if kpis['documentation_coverage_pct'] >= 50 else "🔴 Crítico")
    },
    {
        "Dimensão": "Perfilamento no Knowledge Catalog",
        "Métrica Observada": f"{kpis['profile_coverage_pct']}% ({kpis['profiled_tables_count']}/{kpis['total_tables'] + kpis['total_views']})",
        "Meta de Mercado": "100% das Tabelas Gold/Silver",
        "Status": "🟢 Conforme" if kpis['profile_coverage_pct'] >= 90 else "🟡 Requer Scans"
    },
    {
        "Dimensão": "Grafos Semânticos (GQL)",
        "Métrica Observada": f"{kpis['total_property_graphs']} grafos registrados",
        "Meta de Mercado": ">= 1 Property Graph",
        "Status": "🟢 Habilitado" if kpis['total_property_graphs'] > 0 else "⚪ Não Configurado"
    },
    {
        "Dimensão": "Data Agents (Agentes Conversacionais)",
        "Métrica Observada": f"{kpis['total_data_agents']} agentes ativos",
        "Meta de Mercado": "Multi-Agente por Domínio",
        "Status": "🟢 Ativo" if kpis['total_data_agents'] > 0 else "⚪ Nenhum Agente"
    },
    {
        "Dimensão": "Modelos de IA (Vertex AI + BQML)",
        "Métrica Observada": f"{kpis['total_vertex_models'] + kpis['total_bqml_models']} modelos",
        "Meta de Mercado": "Centralizado no Model Registry",
        "Status": "🟢 Ativo" if (kpis['total_vertex_models'] + kpis['total_bqml_models']) > 0 else "⚪ Nenhum Modelo"
    }
]

df_scorecard = pd.DataFrame(scorecard_data)
display(HTML(df_scorecard.to_html(classes='table table-striped table-hover', index=False)))

### 📊 7. Dashboard Visual de Maturidade & Inventário da Organização

In [ ]:
# @title Gráficos Executivos Globais
fig, axs = plt.subplots(2, 2, figsize=(16, 11))

# 1. Donut: Cobertura de Colunas
doc_cols = kpis['documented_columns_count']
undoc_cols = max(0, kpis['total_columns'] - doc_cols)
axs[0, 0].pie(
    [doc_cols, undoc_cols],
    labels=['Colunas Documentadas', 'Sem Descrição'],
    colors=['#2ecc71', '#e74c3c'],
    autopct='%1.1f%%',
    startangle=140,
    wedgeprops=dict(width=0.4, edgecolor='w')
)
axs[0, 0].set_title('Maturidade Semântica da Org: Cobertura de Colunas (Grounding IA)', fontweight='bold')

# 2. Bar: Ativos de IA
ai_categories = ['Vertex Models', 'Vertex Datasets', 'Data Agents', 'Property Graphs', 'BQML Models']
ai_values = [
    kpis['total_vertex_models'],
    len(manifest['vertex_ai']['datasets']),
    kpis['total_data_agents'],
    kpis['total_property_graphs'],
    kpis['total_bqml_models']
]
bars = axs[0, 1].bar(ai_categories, ai_values, color=['#4285F4', '#34A853', '#FBBC05', '#EA4335', '#9b59b6'])
axs[0, 1].set_title('Inventário Global de IA e Análise Conversacional', fontweight='bold')
for bar in bars:
    yval = bar.get_height()
    axs[0, 1].text(bar.get_x() + bar.get_width()/2.0, yval + 0.5, int(yval), ha='center', va='bottom', fontweight='bold')

# 3. Donut: Perfilamento Dataplex
profiled = kpis['profiled_tables_count']
not_profiled = (kpis['total_tables'] + kpis['total_views']) - profiled
axs[1, 0].pie(
    [max(0, profiled), max(0, not_profiled)],
    labels=['Tabelas Perfiladas', 'Sem Data Profile'],
    colors=['#1abc9c', '#f39c12'],
    autopct='%1.1f%%',
    startangle=140,
    wedgeprops=dict(width=0.4, edgecolor='w')
)
axs[1, 0].set_title('Knowledge Catalog: Scans de Perfil de Dados (Dataplex)', fontweight='bold')

# 4. Bar: Pipelines & Bancos
pipeline_cats = ['Dataflow Jobs', 'Datastream CDC', 'Composer Envs', 'Cloud SQL', 'Spanner']
pipeline_vals = [
    kpis['total_dataflow_jobs'],
    kpis['total_datastream_streams'],
    kpis['total_composer_environments'],
    kpis['total_cloud_sql_instances'],
    kpis['total_spanner_instances']
]
bars2 = axs[1, 1].barh(pipeline_cats, pipeline_vals, color='#34495e')
axs[1, 1].set_title('Infraestrutura de Ingestão, Carga e Bancos na Organização', fontweight='bold')
for bar in bars2:
    xval = bar.get_width()
    axs[1, 1].text(xval + 0.3, bar.get_y() + bar.get_height()/2.0, int(xval), ha='left', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 🔍 8. Dicionário de Dados Tabular com Project ID (`data_catalog_dictionary.csv`)
Inspecione as colunas catalogadas. Cada linha indica explicitamente o **`project_id`**, `dataset_id`, tabela e descrição.

In [ ]:
# @title Visualizar Dicionário de Dados Multi-Projeto
df_dict = pd.read_csv(f"{OUTPUT_DIR}/data_catalog_dictionary.csv")
print(f"📊 Total de Colunas Catalogadas na Organização: {len(df_dict)}")
display(df_dict[['project_id', 'dataset_id', 'table_name', 'column_name', 'data_type', 'column_description', 'dataplex_profile_scan_active']].head(15))

# Resumo por Projeto e Dataset
if not df_dict.empty:
    df_ds = df_dict.groupby(['project_id', 'dataset_id']).agg(
        tabelas=('table_name', 'nunique'),
        colunas=('column_name', 'count'),
        colunas_documentadas=('is_column_documented', 'sum')
    ).reset_index()
    df_ds['pct_documentado'] = (df_ds['colunas_documentadas'] / df_ds['colunas'] * 100).round(2)
    print('\n📈 Resumo de Cobertura por Projeto e Dataset:')
    display(df_ds)

### 💬 9. Catálogo de Agentes Conversacionais (Data Agents) por Projeto

In [ ]:
# @title Lista de Data Agents Cadastrados na Organização
agents = manifest['data_agents']['data_agents']
print(f"💬 Total de Data Agents Encontrados: {len(agents)}\n")

for a in agents[:8]:
    print(f"🤖 Agente: {a['display_name']} ({a['agent_id']}) | Projeto: `{a.get('project_id', 'N/A')}`")
    print(f"   Descrição: {a.get('description', 'N/A')}")
    print(f"   Fontes Vinculadas: {', '.join(a['referenced_tables'] + a['referenced_property_graphs'])}")
    print(f"   Perguntas Validadas ({a['verified_queries_count']}):")
    for q in a.get('verified_queries', [])[:2]:
        print(f"      - \"{q['question']}\"")
    print('-'*80)

### 💼 10. Business Assessment: Casos de Negócio & Quick Wins Identificados
Abaixo estão os Casos de Negócio corporativos formulados automaticamente a partir da análise arquitetural e semântica dos metadados.

In [ ]:
# @title 💼 Tabela Executiva de Casos de Negócio por Domínio
biz_data = manifest.get('business_assessment', {})
cases = biz_data.get('business_cases', [])
header_biz = biz_data.get('business_assessment_header', {})

print(f"💼 Total de Casos de Negócio Estruturados: {len(cases)}")
print(f"🟢 Quick Wins Identificados (Alto Impacto / Baixa Complexidade): {header_biz.get('quick_wins_count', 0)}")
print(f"🔵 Apostas Estratégicas (Alto Impacto / Alta Complexidade): {header_biz.get('strategic_bets_count', 0)}\n")

if cases:
    df_cases = pd.DataFrame([
        {
            "ID": c["case_id"],
            "Domínio": c["domain"],
            "Caso de Negócio": c["title"],
            "Entregável (Módulo)": c["deliverable"],
            "Impacto": c["impact_level"],
            "Complexidade": c["complexity_level"],
            "Prazo": c["timeframe_weeks"],
            "Quadrante": c["quadrant"],
            "Prontidão IA": c["readiness_status"]
        }
        for c in cases
    ])
    display(HTML(df_cases.to_html(classes='table table-striped table-hover', index=False)))
else:
    print("Nenhum caso de negócio específico gerado.")

### 🎯 11. Matriz de Priorização Executiva (Quick Wins vs. Apostas Estratégicas)
Visualização cartesiana em 4 quadrantes para direcionamento C-Level e priorização das iniciativas de dados e IA.

In [ ]:
# @title 🎯 Matriz de Priorização (Impacto no Negócio vs. Complexidade Técnica)
import numpy as np

if cases:
    fig, ax = plt.subplots(figsize=(13, 7))
    impact_map = {'BAIXO': 1, 'MÉDIO': 2, 'ALTO': 3}
    complexity_map = {'BAIXA': 1, 'MÉDIA': 2, 'ALTA': 3}
    quadrant_colors = {
        'Quick Win': '#2ecc71',
        'Aposta Estratégica': '#3498db',
        'Melhoria Tática': '#f39c12',
        'Longo Prazo': '#95a5a6'
    }

    for c in cases:
        x = complexity_map.get(c['complexity_level'], 2) + np.random.uniform(-0.12, 0.12)
        y = impact_map.get(c['impact_level'], 2) + np.random.uniform(-0.12, 0.12)
        col = quadrant_colors.get(c['quadrant'], '#34495e')
        ax.scatter(x, y, color=col, s=300, alpha=0.9, edgecolors='black', linewidth=1.5, zorder=5)
        label_text = f"{c['case_id']}: {c['title'][:28]}..."
        ax.annotate(
            label_text,
            (x, y),
            xytext=(8, 6),
            textcoords='offset points',
            fontsize=9,
            fontweight='bold',
            color='#2c3e50',
            bbox=dict(boxstyle='round,pad=0.25', fc='white', ec=col, alpha=0.9)
        )

    # Linhas de demarcação dos 4 quadrantes
    ax.axvline(2, color='#bdc3c7', linestyle='--', linewidth=1.5)
    ax.axhline(2, color='#bdc3c7', linestyle='--', linewidth=1.5)

    # Textos dos Quadrantes
    ax.text(1.05, 2.85, '🟢 QUADRANTE 1: QUICK WINS\n(Alto Impacto / Baixa Complexidade)', fontsize=11, fontweight='bold', color='#27ae60')
    ax.text(2.05, 2.85, '🔵 QUADRANTE 2: APOSTAS ESTRATÉGICAS\n(Alto Impacto / Alta Complexidade)', fontsize=11, fontweight='bold', color='#2980b9')
    ax.text(1.05, 1.15, '🟡 QUADRANTE 3: MELHORIAS TÁTICAS\n(Médio Impacto / Baixa Complexidade)', fontsize=11, fontweight='bold', color='#d35400')
    ax.text(2.05, 1.15, '⚪ QUADRANTE 4: LONGO PRAZO\n(Baixo Impacto / Alta Complexidade)', fontsize=11, fontweight='bold', color='#7f8c8d')

    ax.set_xlim(0.7, 3.3)
    ax.set_ylim(0.7, 3.3)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(['Baixa', 'Média', 'Alta'], fontsize=12, fontweight='bold')
    ax.set_yticks([1, 2, 3])
    ax.set_yticklabels(['Baixo', 'Médio', 'Alto'], fontsize=12, fontweight='bold')
    ax.set_xlabel('Complexidade Técnica de Implementação', fontsize=12, fontweight='bold')
    ax.set_ylabel('Impacto no Negócio (ROI & Eficiência)', fontsize=12, fontweight='bold')
    ax.set_title('Matriz de Priorização: Casos de Negócio & Quick Wins GCP', fontsize=14, fontweight='bold')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.tight_layout()
    plt.show()
else:
    print('Nenhum dado disponível para matriz de priorização.')

### 🌊 12. Roadmap Estratégico por Ondas de Implementação (Ondas 1, 2 e 3)
Sequenciamento temporal recomendado das iniciativas para maximizar retorno ágil e maturidade de dados.

In [ ]:
# @title 🌊 Roadmap Estratégico por Ondas
roadmap = biz_data.get('wave_roadmap', {})
for wave_key, wave_info in roadmap.items():
    print('='*80)
    print(f"📍 {wave_info['phase'].upper()} | {wave_info['timeframe']}")
    print('='*80)
    for c in wave_info.get('cases', []):
        print(f"🎯 [{c['case_id']}] {c['title']} ({c['timeframe_weeks']})")
        print(f"   🏢 Domínio: {c['domain']} | Quadrante: {c['quadrant']}")
        print(f"   💡 Entregável: {c['deliverable']}")
        print(f"   💰 Impacto: {c['business_impact']}")
        print(f"   🤖 Prontidão IA: {c['readiness_status']} (Score: {c['readiness_score']} %)")
        print(f"   🏆 Benchmark: {c['benchmark']}")
        print(f"   ☁️ Stack GCP: {c['target_gcp_stack']}")
        print('-' * 60)
    print()

### ☁️ 13. Confirmação dos Arquivos no Google Cloud Storage (GCS) & Entrega Final
Verifique abaixo os arquivos consolidados da organização salvos localmente e no GCS.

In [ ]:
# @title Resumo dos Artefatos de Saída
import os, glob
zip_candidates = glob.glob(f"{OUTPUT_DIR}/metadata_assessment_*.zip")
zip_path = zip_candidates[0] if zip_candidates else f"{OUTPUT_DIR}/metadata_assessment_organization.zip"
zip_filename = os.path.basename(zip_path)
manifest_path = f"{OUTPUT_DIR}/metadata_assessment_manifest.json"
dict_path = f"{OUTPUT_DIR}/data_catalog_dictionary.csv"
summary_path = f"{OUTPUT_DIR}/executive_assessment_summary.md"
biz_md_path = f"{OUTPUT_DIR}/business_assessment_cases.md"
biz_json_path = f"{OUTPUT_DIR}/business_assessment_cases.json"

print("Arquivos Gerados com Sucesso:")
if os.path.exists(zip_path):
    print(f"📦 1. Pacote ZIP Completo: {zip_path} ({round(os.path.getsize(zip_path)/1024, 2)} KB)")
if os.path.exists(biz_md_path):
    print(f"💼 2. {biz_md_path} ({round(os.path.getsize(biz_md_path)/1024, 2)} KB) -> Casos de Negócio & Quick Wins")
if os.path.exists(biz_json_path):
    print(f"📊 3. {biz_json_path} ({round(os.path.getsize(biz_json_path)/1024, 2)} KB) -> JSON Estruturado de Negócio")
print(f"📄 4. {manifest_path} ({round(os.path.getsize(manifest_path)/1024, 2)} KB) -> Manifesto Técnico Canônico")
print(f"📊 5. {dict_path} ({round(os.path.getsize(dict_path)/1024, 2)} KB) -> Dicionário de Colunas")
print(f"📑 6. {summary_path} ({round(os.path.getsize(summary_path)/1024, 2)} KB) -> Sumário Executivo de Governança")

if GCS_OUTPUT_URI.strip().startswith('gs://'):
    print(f"\n☁️ Artefatos e Pacote ZIP sincronizados no GCS: {GCS_OUTPUT_URI}")
    print(f"   - {GCS_OUTPUT_URI.rstrip('/')}/{zip_filename}")
    print("Você pode consultar e baixar os arquivos diretamente pelo Cloud Storage Console ou gsutil.")

# Exibe preview do relatório de Casos de Negócio inline
target_preview = biz_md_path if os.path.exists(biz_md_path) else summary_path
if os.path.exists(target_preview):
    with open(target_preview, 'r', encoding='utf-8') as f:
        content = f.read()
    display(Markdown(content[:2000] + "\n\n*(... relatório completo gerado no arquivo)*"))

### 📥 14. Download Imediato do Pacote ZIP (1 Clique no Navegador)
Execute a célula abaixo para baixar automaticamente o pacote ZIP consolidado com todos os resultados no seu computador.

In [ ]:
# @title 🗜️ Download do Pacote ZIP (.zip)
import os, glob
zip_candidates = glob.glob(f"{OUTPUT_DIR}/metadata_assessment_*.zip")
zip_path = zip_candidates[0] if zip_candidates else f"{OUTPUT_DIR}/metadata_assessment_organization.zip"

if os.path.exists(zip_path):
    print(f"🗜️ Pacote ZIP pronto: {zip_path} ({round(os.path.getsize(zip_path)/1024, 2)} KB)")
    try:
        from google.colab import files
        print("Iniciando download automático no seu navegador...")
        files.download(zip_path)
    except Exception:
        try:
            from IPython.display import FileLink
            display(FileLink(zip_path))
        except Exception:
            print(f"📁 Arquivo ZIP salvo em: {os.path.abspath(zip_path)}")
else:
    print(f"⚠️ Arquivo ZIP não encontrado em {zip_path}.")